# ⚗️ 41 · Perturbation biology and causal inference
### Potential outcomes, causal graphs and adjustment, instrumental variables, pooled CRISPR screens, Perturb-seq, learning gene networks from interventions and predicting unseen perturbations — with every true causal effect known

**BiologyToML** · Level 2 · Research frontier · Part I — Causality, reliability and AI-driven discovery · Module 41 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/41_perturbation_causal_inference.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (metrics), PyTorch (a small MLP in Section 7) |
| **Prerequisites** | Level 1 (especially Modules 06, 09, 13 and 15–18); [Module 24](24_glm_count_data.ipynb) and [Module 30](30_single_cell_rnaseq.ipynb) help; basic PyTorch (Module 22) for Section 7 |
| **Estimated time** | 185 min (reading + running + the ✏️ exercises) |
| **Builds on** | [06 · Linear regression](06_linear_regression_gwas.ipynb), [09 · Logistic regression](09_logistic_regression.ipynb), [13 · Mixture models](13_mixture_models_em.ipynb), [15](15_bayesian_networks_representation.ipynb)–[17 · Bayesian networks](17_bayesian_networks_inference_learning.ipynb), [18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), [23 · Model selection](23_model_selection.ipynb), [24 · GLMs for counts](24_glm_count_data.ipynb), [35 · Statistical genetics](35_statistical_genetics.ipynb) |
| **Research frontier covered** | potential outcomes and causal graphs, confounding and adjustment, instrumental variables, CRISPR screens, Perturb-seq, learning gene networks from interventions, predicting unseen perturbations — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** Sections 1–3 use small cohort simulators written in this notebook (a treatment confounded
> by disease severity, a continuous confounder, a genetic instrument); because both potential outcomes or the true
> effect are generated, every estimate can be compared with the truth. Section 4 simulates a pooled CRISPR knockout
> screen with known essential genes. Sections 5–7 use the toolkit's `simulate_perturb_seq`: a CRISPR-interference
> Perturb-seq screen on a hidden 40-gene regulatory network (a DAG with saturating dynamics), with guide escape and
> double knockdowns; the true network, the true effect of every perturbation and which cells escaped are known. Real
> screens add thousands of genes, off-target effects, feedback loops and cell-state heterogeneity — see Section 8.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** causal effects with potential outcomes (individual and average treatment effects), **explain** why randomisation identifies them, and **provoke** Simpson's paradox in an observational cohort.
2. **Read** a causal graph: apply the back-door criterion and the adjustment formula; **implement** stratification, regression adjustment, inverse-probability weighting and a doubly robust estimator; **diagnose** positivity violations and collider bias.
3. **Estimate** a causal effect with instrumental variables (Wald ratio, two-stage least squares), **show** how weak instruments fail, and **connect** IV to Mendelian randomisation and to imperfect compliance.
4. **Simulate and analyse** a pooled CRISPR knockout screen: guide log fold changes, count noise, robust gene-level aggregation (RRA); **measure** precision and recall for the true essential genes.
5. **Analyse** Perturb-seq data: normalisation artefacts, per-perturbation effects against the truth, and guide-escape detection with a mixture model (the Mixscape idea).
6. **Learn** a gene network from observational and interventional cells — a GGM skeleton, Markov equivalence, descendant sets, transitive reduction — and **grade** it against the true DAG.
7. **Predict** unseen perturbations honestly: an additive baseline against a learned model for double knockdowns, and an adjustment-based prediction of an unmeasured single knockdown.
8. **Connect** the methods to MAGeCK, Mixscape, GEARS, CPA and genome-scale Perturb-seq, and **name** what can and cannot be predicted.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Correlation vs causation | potential outcomes, randomisation, Simpson's paradox; 🧪 vs the true ATE | 20 min |
| 2 · Causal graphs and adjustment | back-door criterion, adjustment, IPW, doubly robust, positivity, colliders; 🧪 | 30 min |
| 3 · Instrumental variables | Wald ratio, 2SLS, weak instruments, compliance; 🧪 | 20 min |
| 4 · Pooled CRISPR screens | a screen simulator, guide LFCs, RRA; 🧪 precision/recall for essential genes | 25 min |
| 5 · Perturb-seq | normalisation, effects, guide escape with a mixture; 🧪 vs the true effects | 25 min |
| 6 · Learning the network | observational skeleton vs interventions; 🧪 precision/recall vs the true DAG | 20 min |
| 7 · Predicting unseen perturbations | doubles: additive vs learned; unseen singles by adjustment; 🧪 | 20 min |
| 8 · 🧭 From toy to real data | Replogle, Norman, tools, limits | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

Patients who receive a new drug die more often than those who do not. Does the drug kill? Or do doctors give it to
the sickest patients? Cells with high expression of gene A also express gene B highly. Does A switch on B, B switch on
A, or does a third factor drive both? Observational data — even millions of single cells — show **associations**.
Biology and medicine need **causes**: what happens if we *intervene*, with a drug, a knockout, an inhibitor?

Two developments changed what is possible. First, a mature **theory of causal inference** — potential outcomes
(Neyman, Rubin) and causal graphs (Pearl) — says precisely when an association can be read causally and how to adjust
for confounding when it cannot. Second, **CRISPR** made interventions cheap and scalable: pooled screens knock out every
gene in the genome and measure which knockouts change cell fitness (Shalem *et al.* 2014), and **Perturb-seq** reads out
a whole transcriptome for every perturbed cell (Dixit *et al.* 2016; Adamson *et al.* 2016). Replogle *et al.* (2022)
applied CRISPR interference to essentially every expressed gene of a human cell line and profiled millions of cells;
Norman *et al.* (2019) activated genes singly and in pairs to map genetic interactions.

Our simulations mimic both levels. A clinical cohort and a genetic study (Sections 1–3) teach the logic of
adjustment and instruments with a known answer. A pooled screen with known essential genes (Section 4) and a
Perturb-seq experiment on a hidden 40-gene network (Sections 5–7) — with guide escape, double knockdowns and saturating
dynamics — let us ask the questions of the field:

1. When does an association measure a causal effect, and how do we fix it when it does not?
2. Which genes are essential, and which genes does each knockdown affect?
3. Can we recover the regulatory network — its *directions* — from perturbation data?
4. Can we predict the effect of perturbations we have **not** done?

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import average_precision_score, precision_recall_curve

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_perturb_seq` generates the Perturb-seq screen with its hidden network and true effects, `log_normalize` is the total-count log transform of Module 30, `benjamini_hochberg` adjusts p-values, `sigmoid` is the logistic function, `draw_graph` draws small causal graphs, and `precision_recall_f1` grades binary calls. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_perturb_seq(rng: np.random.Generator, n_genes: int = 40, n_targets: int = 15, n_pairs: int = 12,
                         cells_per_perturbation: int = 80, n_control: int = 800, edge_prob: float = 0.08,
                         knockdown: float = 2.0, escape_rate: float = 0.1, noise: float = 0.3,
                         depth: float = 3000) -> dict:
    """Simulate a CRISPR-interference Perturb-seq screen on a known gene-regulatory network.

    The network is a random DAG: each gene is regulated by earlier genes (probability
    ``edge_prob`` per pair) with signed weights.  A cell's deviation from baseline is
    d_g = c·tanh((Σ_parents w·d_parent + ε_g)/c), i.e. linear for small changes but
    *saturating* for large ones, so double knockdowns are not simply the sum of two single
    ones.  A knockdown clamps the target at −``knockdown`` (an intervention, do(X = x)); a
    fraction ``escape_rate`` of cells keep a working gene (guide escape).  Perturbations: one
    control group, ``n_targets`` single knockdowns and ``n_pairs`` double knockdowns.

    Returns a dict with ``expression`` (log scale, cells × genes), ``counts`` (UMIs), ``obs``
    (perturbation label, e.g. "ctrl", "G07", "G07+G12"), ``var``, the true weighted
    adjacency ``W`` (W.loc[parent, child]) and ``true_effect`` (noise-free mean shift per
    perturbation).
    """
    genes = np.array([f"G{j:02d}" for j in range(1, n_genes + 1)], dtype=object)
    order = rng.permutation(n_genes)                                     # hidden topological order
    W = np.zeros((n_genes, n_genes))
    for a in range(n_genes):
        for b in range(a + 1, n_genes):
            if rng.random() < edge_prob:
                W[order[a], order[b]] = rng.choice([-1, 1]) * rng.uniform(0.6, 1.2)
    has_children = np.flatnonzero((W != 0).sum(axis=1) > 0)
    pool = has_children if len(has_children) >= n_targets else np.arange(n_genes)
    targets = rng.choice(pool, n_targets, replace=False)
    pairs = set()
    while len(pairs) < n_pairs:
        a, b = sorted(rng.choice(targets, 2, replace=False))
        pairs.add((a, b))
    conditions = [()] + [(t,) for t in targets] + sorted(pairs)
    sizes = [n_control] + [cells_per_perturbation] * (len(conditions) - 1)
    base = rng.normal(1.0, 1.0, n_genes)                                 # baseline log expression
    sat = 1.5                                                            # saturation scale c

    def propagate(clamp, eps):
        d = np.zeros_like(eps)
        for g in order:
            if g in clamp:
                d[:, g] = np.where(clamp[g], -knockdown, sat * np.tanh((d @ W[:, g] + eps[:, g]) / sat))
            else:
                d[:, g] = sat * np.tanh((d @ W[:, g] + eps[:, g]) / sat)
        return d

    expr, labels, effects = [], [], {}
    for cond, n in zip(conditions, sizes):
        label = "ctrl" if not cond else "+".join(genes[list(cond)])
        clamp = {g: rng.random(n) >= escape_rate for g in cond}
        d = propagate(clamp, rng.normal(0, noise, (n, n_genes)))
        expr.append(base + d)
        labels += [label] * n
        effects[label] = propagate({g: np.ones(1, bool) for g in cond}, np.zeros((1, n_genes)))[0]
    expression = np.vstack(expr)
    prob = np.exp(expression) / np.exp(expression).sum(axis=1, keepdims=True)
    counts = rng.poisson(depth * np.exp(rng.normal(0, 0.2, len(labels)))[:, None] * prob).astype(np.int32)
    obs = pd.DataFrame({"perturbation": labels,
                        "n_targets": [0 if lab == "ctrl" else lab.count("+") + 1 for lab in labels]},
                       index=[f"cell{i:05d}" for i in range(len(labels))])
    var = pd.DataFrame({"baseline": base, "is_target": np.isin(np.arange(n_genes), targets),
                        "topological_rank": np.argsort(order)}, index=genes)
    return dict(expression=expression, counts=counts, obs=obs, var=var,
                W=pd.DataFrame(W, index=genes, columns=genes),
                true_effect=pd.DataFrame(effects, index=genes).T)


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_perturb_seq', 'sigmoid', 'confusion_counts', 'precision_recall_f1', 'log_normalize', 'benjamini_hochberg', 'draw_graph']))

In [ ]:
# ── 0.1 Threads and clock ──
torch.set_num_threads(min(2, torch.get_num_threads()))
T_START = time.time()
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

---
## 1 · Correlation vs causation

> 📍 **Where we are.** Every model of Level 1 learned *associations* — $P(Y \mid X)$. Here we define what a *causal*
> effect is, and see an association point in exactly the wrong direction.

🧮 **Math refresher — potential outcomes.** For each patient $i$ imagine two outcomes: $Y_i(1)$ if treated and $Y_i(0)$
if not. The **individual treatment effect** is $Y_i(1) - Y_i(0)$; the **average treatment effect** is

$$\text{ATE} = \mathbb E[\,Y(1) - Y(0)\,] = \mathbb E[Y(1)] - \mathbb E[Y(0)] .$$

*In words:* the ATE compares the whole population treated with the whole population untreated — two worlds of which we
can observe only one per person. The **fundamental problem of causal inference**: for every patient we see either
$Y_i(1)$ or $Y_i(0)$, never both. What we can compute from data is the **association**

$$\mathbb E[Y \mid T = 1] - \mathbb E[Y \mid T = 0] = \mathbb E[Y(1) \mid T = 1] - \mathbb E[Y(0) \mid T = 0] ,$$

*In words:* the outcome of the people who *were* treated minus that of the people who were *not*. The two coincide when
treatment is independent of the potential outcomes, $T \perp \{Y(0), Y(1)\}$ — exactly what **randomisation**
guarantees. Without it, treated and untreated people may differ in ways that affect the outcome: **confounding**.

Our simulator gives every patient *both* potential outcomes, so we know the true ATE. Doctors give the drug mostly to
severe cases (80 % of severe vs 20 % of mild patients); the drug raises the recovery probability in both groups.

In [ ]:
# ── 1.1 A SIMULATED cohort in which every patient has both potential outcomes ──
def simulate_cohort(rng, n=4000, randomised=False):
    """Mild or severe disease; a latent 'robustness' u decides recovery with (Y1) and without (Y0) the drug.
    Observationally the drug goes mostly to severe patients; in a trial a fair coin decides."""
    severe = rng.random(n) < 0.4
    u = rng.random(n)
    y0 = u < np.where(severe, 0.35, 0.85)                  # recovers without the drug
    y1 = u < np.where(severe, 0.50, 0.92)                  # recovers with it (the drug never harms)
    p_treat = np.full(n, 0.5) if randomised else np.where(severe, 0.8, 0.2)
    treated = rng.random(n) < p_treat
    return pd.DataFrame({"severe": severe, "treated": treated, "Y0": y0, "Y1": y1, "Y": np.where(treated, y1, y0)})


cohort = simulate_cohort(np.random.default_rng(SEED + 1))
true_ate = (cohort.Y1.astype(int) - cohort.Y0.astype(int)).mean()
naive = cohort.Y[cohort.treated].mean() - cohort.Y[~cohort.treated].mean()
print(f"true ATE (from both potential outcomes): {true_ate:+.3f}")
print(f"naive association, recovery(treated) − recovery(untreated): {naive:+.3f}")
rows = {}
for name, d in [("mild", cohort[~cohort.severe]), ("severe", cohort[cohort.severe]), ("all patients", cohort)]:
    r_u, r_t = d.Y[~d.treated].mean(), d.Y[d.treated].mean()
    rows[name] = {"n untreated": int((~d.treated).sum()), "n treated": int(d.treated.sum()),
                  "recovery untreated": r_u, "recovery treated": r_t, "difference": r_t - r_u}
print("\nrecovery by severity and treatment:")
print(pd.DataFrame.from_dict(rows, orient="index").round(3).to_string())

In [ ]:
# ── 1.2 🧪 200 observational studies vs 200 randomised trials ──
rng = np.random.default_rng(SEED + 2)
est = {"observational": [], "randomised": []}
for _ in range(200):
    for kind in est:
        d = simulate_cohort(rng, randomised=(kind == "randomised"))
        est[kind].append(d.Y[d.treated].mean() - d.Y[~d.treated].mean())
fig, ax = plt.subplots(figsize=(7.5, 3.0))
for k, (kind, v) in enumerate(est.items()):
    ax.hist(v, bins=30, alpha=0.7, color=CLASS_COLORS[k], label=f"{kind}: mean {np.mean(v):+.3f}")
ax.axvline(true_ate, color="k", ls="--", label=f"true ATE {true_ate:+.3f}")
ax.set_xlabel("difference in recovery rate, treated − untreated"); ax.set_ylabel("studies"); ax.legend(fontsize=7)
ax.set_title("SIMULATED: the same drug, two study designs")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The drug helps everyone — by design it raises the recovery probability by 7 points in mild
and 15 points in severe patients, an ATE of +0.10 — yet treated patients recover *less* often than untreated ones
(−0.18). Within each severity group the drug looks good (+0.06 and +0.12 in this sample); pooled, it looks harmful. This reversal is **Simpson's paradox** (Simpson 1951): the
treated group is dominated by severe patients, who recover less with or without the drug. Severity is a **confounder**
— a common cause of treatment and outcome.

The histograms make it systematic: observational studies are consistently, confidently wrong (mean −0.16, none
anywhere near the truth), while randomised trials scatter around the true ATE (mean +0.10). Randomisation does not make patients alike; it makes *treatment
independent of everything about them*, measured or not.

⚠️ **Which table is right — pooled or stratified?** Simpson's paradox has no purely statistical resolution: the same
numbers can arise from different causal structures, and the right answer depends on the causal graph (Section 2). Here
severity affects treatment, so we must stratify. If the stratifying variable were *caused by* the treatment, stratifying
would be wrong.

✏️ **Your turn.** Make the drug harmful for mild patients (`y1 = u < np.where(severe, 0.50, 0.80)`). Compute the true
ATE, the naive difference and the stratum-specific differences. Can the naive analysis now make a harmful drug look
helpful, or a helpful one harmful?

---
## 2 · Causal graphs and adjustment

> 📍 **Where we are.** We cannot always randomise. Causal graphs tell us which variables to adjust for, and
> estimators turn that into numbers. We build four of them and grade them against the truth.

> 🔗 **Builds on** [Module 15](15_bayesian_networks_representation.ipynb) (a DAG factorises a joint distribution),
> [Module 16](16_bayesian_networks_independence.ipynb) (d-separation: chains and forks are blocked by conditioning,
> colliders are opened by it) and [Module 17](17_bayesian_networks_inference_learning.ipynb), Section 10 (an intervention
> cuts the arrows into the perturbed node).

**Interventions as graph surgery.** Pearl's **do-operator** $P(Y \mid do(T = t))$ is the distribution of $Y$ in a world
where we *set* $T$ to $t$ for everyone — we delete all arrows into $T$ and keep the rest of the graph. It differs from
the conditional $P(Y \mid T = t)$, which describes the people who *happened* to have $T = t$.

**The back-door criterion.** A set of measured variables $S$ is a valid adjustment set for the effect of $T$ on $Y$ if
(i) no variable in $S$ is a descendant of $T$, and (ii) $S$ blocks every path from $T$ to $Y$ that starts with an arrow
*into* $T$ (a "back door"). Then

$$P(Y \mid do(T = t)) = \sum_{s} P(Y \mid T = t, S = s)\,P(S = s) .$$

*In words:* estimate the treatment–outcome relation *within* each level of $S$ (where the back door is closed), then
average over the population distribution of $S$ — not over its distribution among the treated. This **adjustment
formula** is what stratification did in Section 1.

In [ ]:
# ── 2.1 Three small causal graphs ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
draw_graph([("S", "T"), ("S", "Y"), ("T", "Y")], {"S": (0.5, 0.9), "T": (0.1, 0.15), "Y": (0.9, 0.15)}, ax=axes[0],
           highlight={"S"}, radius=0.09, title="confounding: adjust for S\n(S = severity, T = drug, Y = recovery)")
draw_graph([("T", "M"), ("M", "Y"), ("T", "Y")], {"T": (0.1, 0.15), "M": (0.5, 0.9), "Y": (0.9, 0.15)}, ax=axes[1],
           radius=0.09, title="mediation: do NOT adjust for M\nif you want the total effect")
draw_graph([("A", "H"), ("B", "H")], {"A": (0.1, 0.9), "B": (0.9, 0.9), "H": (0.5, 0.15)}, ax=axes[2],
           highlight={"H"}, radius=0.09, title="collider: selecting on H\ncreates an A–B association")
plt.tight_layout(); plt.show()

# the adjustment formula on the Section 1 cohort
p_s = cohort.severe.mean()
within = cohort.groupby(["severe", "treated"]).Y.mean()
adjusted = sum((within[(s, True)] - within[(s, False)]) * (p_s if s else 1 - p_s) for s in (False, True))
print(f"Section 1 cohort — naive {naive:+.3f}, adjustment formula {adjusted:+.3f}, true ATE {true_ate:+.3f}")

### A continuous confounder and four estimators

Real confounders are continuous and many. We simulate a severity score $X$ (higher = sicker); the probability of
treatment rises with $X$ (the **propensity score** $e(x) = P(T = 1 \mid X = x)$), and the outcome — say, a lung-function
score after a week — falls *non-linearly*, ever more steeply in the sickest patients, and rises by $\tau = 1$ with
treatment:

$$Y = \tau T - 0.8\,e^{X} + \varepsilon, \qquad P(T = 1 \mid X) = \sigma(-0.3 + \beta X) .$$

*In words:* sicker patients are treated more often and do worse, so the naive comparison will be biased downwards;
$\beta$ sets the strength of confounding. The estimators:

1. **Stratification** — the adjustment formula with $X$ cut into five quintiles.
2. **Regression adjustment** — fit $Y \sim 1 + T + X$ by least squares and read off the coefficient of $T$
   ([Module 06](06_linear_regression_gwas.ipynb)). Valid if the outcome model is right; ours is deliberately linear.
3. **Inverse-probability weighting (IPW)** (Rosenbaum & Rubin 1983 introduced the propensity score) — fit $e(x)$ by
   logistic regression ([Module 09](09_logistic_regression.ipynb)) and weight each treated patient by $1/\hat e$, each
   untreated one by $1/(1 - \hat e)$:

   $$\widehat{\text{ATE}}_{\text{IPW}} = \frac{\sum_i T_i Y_i / \hat e_i}{\sum_i T_i / \hat e_i} - \frac{\sum_i (1 - T_i) Y_i / (1 - \hat e_i)}{\sum_i (1 - T_i)/(1 - \hat e_i)} .$$

   *In words:* a treated patient who was unlikely to be treated stands in for many similar untreated ones; the weights
   rebuild, in each arm, a pseudo-population with the covariate distribution of everybody — as if treatment had been
   randomised. Valid if the propensity model is right.
4. **Augmented IPW (AIPW, "doubly robust")** — combine an outcome model $\hat\mu_t(x)$ for each arm with the weights:
   $\widehat{\text{ATE}} = \frac{1}{n}\sum_i \big[\hat\mu_1(x_i) - \hat\mu_0(x_i) + \frac{T_i (Y_i - \hat\mu_1(x_i))}{\hat e_i}
   - \frac{(1 - T_i)(Y_i - \hat\mu_0(x_i))}{1 - \hat e_i}\big]$. *In words:* start from the outcome-model prediction and
   correct it with the weighted residuals; it is consistent if *either* model is right. (We normalise the weights in the
   two correction terms to sum to one, a common stabilisation.)

In [ ]:
# ── 2.2 The confounded study and a logistic regression for the propensity ──
def simulate_confounded(rng, n=2000, beta=1.5, tau=1.0):
    """Severity X, treatment T with propensity sigmoid(-0.3 + beta·X), outcome Y = tau·T − 0.8·exp(X) + noise."""
    x = rng.normal(size=n)
    e = sigmoid(-0.3 + beta * x)
    t = (rng.random(n) < e).astype(float)
    y = tau * t - 0.8 * np.exp(x) + rng.normal(size=n)
    return x, t, y, e


def fit_logistic(X, y, n_iter=25):
    """Logistic regression by Newton–Raphson (Module 09); X must include an intercept column."""
    w = np.zeros(X.shape[1])
    for _ in range(n_iter):
        p = sigmoid(X @ w)
        H = X.T @ (X * (p * (1 - p))[:, None]) + 1e-8 * np.eye(X.shape[1])
        w += np.linalg.solve(H, X.T @ (y - p))
    return w

In [ ]:
# ── 2.3 Five estimates of the ATE from one study ──
def estimate_ate(x, t, y):
    """Naive difference, stratification, regression adjustment, IPW (normalised weights) and AIPW."""
    out = {"naive": y[t == 1].mean() - y[t == 0].mean()}
    strata = np.digitize(x, np.quantile(x, [0.2, 0.4, 0.6, 0.8]))
    both_arms = all(0 < t[strata == s].sum() < (strata == s).sum() for s in range(5))
    out["stratified (5 bins)"] = (sum(np.mean(strata == s) * (y[(strata == s) & (t == 1)].mean() - y[(strata == s) & (t == 0)].mean())
                                      for s in range(5)) if both_arms else np.nan)    # undefined if a bin lacks an arm
    D = np.column_stack([np.ones_like(x), t, x])
    out["regression Y ~ T + X"] = np.linalg.lstsq(D, y, rcond=None)[0][1]
    X1 = np.column_stack([np.ones_like(x), x])
    e = np.clip(sigmoid(X1 @ fit_logistic(X1, t)), 1e-6, 1 - 1e-6)
    w1, w0 = t / e, (1 - t) / (1 - e)
    out["IPW"] = (w1 @ y) / w1.sum() - (w0 @ y) / w0.sum()
    mu1 = X1 @ np.linalg.lstsq(X1[t == 1], y[t == 1], rcond=None)[0]           # linear outcome model per arm
    mu0 = X1 @ np.linalg.lstsq(X1[t == 0], y[t == 0], rcond=None)[0]
    out["AIPW (doubly robust)"] = np.mean(mu1 - mu0) + (w1 @ (y - mu1)) / w1.sum() - (w0 @ (y - mu0)) / w0.sum()
    return out, e


x, t, y, e_true = simulate_confounded(np.random.default_rng(SEED + 3))
one, e_hat = estimate_ate(x, t, y)
print("one SIMULATED study (n = 2000, beta = 1.5, true ATE = 1.0):")
print({k: round(float(v), 3) for k, v in one.items()})
print(f"propensity model check: correlation of fitted and true e(x) = {np.corrcoef(e_hat, e_true)[0, 1]:.4f}")

In [ ]:
# ── 2.4 🧪 Bias and variance over 300 replicate studies, with moderate and with strong confounding ──
rng = np.random.default_rng(SEED + 4)
summary, max_w = {}, {}
for beta in [1.5, 4.0]:
    reps = pd.DataFrame([estimate_ate(*simulate_confounded(rng, beta=beta)[:3])[0] for _ in range(300)])
    summary[f"β = {beta}: bias"] = reps.mean() - 1.0                    # NaN-skipping means and SDs
    summary[f"β = {beta}: SD"] = reps.std()
    print(f"β = {beta}: stratified estimate undefined (a bin with no treated or no untreated patient) in "
          f"{reps['stratified (5 bins)'].isna().mean():.0%} of the studies")
    xs, ts, _, es = simulate_confounded(rng, n=2000, beta=beta)
    max_w[beta] = np.r_[1 / es[ts == 1], 1 / (1 - es[ts == 0])]
print(pd.DataFrame(summary).round(3).to_string())
print("\nlargest inverse-probability weight in one study:", {b: round(float(w.max()), 1) for b, w in max_w.items()},
      "| share of total weight carried by the top 1 % of patients:",
      {b: round(float(np.sort(w)[-len(w) // 100:].sum() / w.sum()), 2) for b, w in max_w.items()})

🔍 **What just happened?** With moderate confounding (β = 1.5) the naive difference is badly biased (bias −1.35: the
treatment *looks* harmful again, although it helps). The adjustment methods behave as their assumptions predict:

- **stratification** removes most of the bias but not all (−0.24): within the top quintile the outcome still falls
  steeply with $X$ and the treated are sicker — residual confounding *within* bins;
- **regression adjustment** is almost unbiased (+0.06) *despite* its wrong, linear outcome model: with good overlap,
  treated and untreated patients cover the same range of $X$, so the misfit largely cancels;
- **IPW** and **AIPW** are essentially unbiased (|bias| < 0.01), at the price of a larger spread (SD 0.16 and 0.20 vs
  0.07 for regression).

With strong confounding (β = 4) the arms barely overlap: almost every severe patient is treated, almost every mild one
is not. This is a violation of **positivity** (overlap): $0 < e(x) < 1$ must hold for every $x$ in the population. The
largest weight in one study grows from about 30 to about 170, the top 1 % of patients carry twice the share of the
weight, and now every method is in trouble: the regression **extrapolates** its wrong linear form into regions where
one arm has no data (bias +0.40); IPW is biased (−0.30) and very noisy (SD 0.56); stratification is undefined in 7 % of
the studies (a bin without treated or untreated patients) and biased when defined; only AIPW stays unbiased, with a
huge spread (SD 0.75). There is simply little information about what treatment does to mild patients, because almost
none were treated.

⚠️ **No estimator rescues unmeasured confounding.** All four methods adjust only for $X$. If doctors also used an
unrecorded judgement of frailty, every estimate would be biased, with no warning from the data. That is why
instruments (Section 3) and experiments (Sections 4–7) matter.

✏️ **Your turn.** Give regression adjustment the correct model (`D = np.column_stack([np.ones_like(x), t, np.exp(x)])`).
Does its bias disappear? Then give IPW a *wrong* propensity model (fit the logistic regression on `x**2` only). Which
estimator is now biased — and is AIPW still fine?

### Collider bias: selecting the wrong cells

Adjustment can also *create* bias. In a collider $A \to H \leftarrow B$, the variables $A$ and $B$ are independent, but
conditioning on $H$ — or selecting individuals by $H$ — makes them dependent ([Module 16](16_bayesian_networks_independence.ipynb),
Section 2.1, simulated a case-only study). The classic example is **Berkson's bias** in hospital-based studies: two
unrelated diseases each lead to hospital admission, so among inpatients a patient without one disease is more likely to
have the other. The same trap awaits in the lab: keeping only cells that pass a quality filter that depends on two
biological variables, or only clones that grew.

In [ ]:
# ── 2.5 Berkson's bias: two independent conditions, one hospital ──
rng = np.random.default_rng(SEED + 5)
a, b = rng.normal(size=20000), rng.normal(size=20000)        # e.g. a lung condition and a heart condition (severity)
admitted = a + b + rng.normal(0, 0.5, 20000) > 1.5           # either can put you in hospital
r_pop, r_hosp = np.corrcoef(a, b)[0, 1], np.corrcoef(a[admitted], b[admitted])[0, 1]
print(f"correlation of the two conditions — whole population: {r_pop:+.3f}; admitted patients only "
      f"({admitted.mean():.0%} of the population): {r_hosp:+.3f}")

🔍 **What just happened?** In the population the two conditions are independent (r ≈ 0); among the admitted patients
they are clearly negatively correlated (r ≈ −0.55) — purely because admission depends on both. A study recruiting
patients from the hospital would "discover" that one condition protects against the other. In causal language, the
selection conditioned on a collider and opened the path $A \to H \leftarrow B$.

### 📄 Research connection — Causal inference frameworks

- **What the field does.** Two formal languages dominate. The **potential-outcomes** framework (Neyman 1923; Rubin 1974) defines effects as
    contrasts between counterfactual outcomes; propensity scores (Rosenbaum & Rubin 1983) and doubly robust estimators
    are its workhorses; Angrist, Imbens & Rubin (1996) connected it to instrumental variables. **Causal graphs and
    structural causal models** (Pearl 2009) express assumptions as DAGs, define interventions by graph surgery and give
    graphical identification criteria (back-door, front-door, do-calculus). The two are largely compatible; Hernán &
    Robins (2020) teach both with an epidemiological flavour, Peters, Janzing & Schölkopf (2017) with a machine-learning
    one.
- **What this notebook builds.** Potential outcomes simulated for every patient (so the true ATE is known), Simpson's paradox, the adjustment
    formula, regression adjustment, IPW and AIPW from scratch with their failure modes (model misspecification, positivity
    violations), and collider bias by selection.
- **Where the frontier is.** Current directions include machine-learning nuisance models with valid inference (double/debiased machine
    learning), heterogeneous effects (which patients benefit?), sensitivity analysis for unmeasured confounding,
    "target trial emulation" with electronic health records, and causal discovery from mixed observational and
    interventional data — Sections 6–7.
- **Read next.** Hernán & Robins 2020, *Causal Inference: What If* (free online); Pearl 2009, *Causality* (2nd ed.); Rosenbaum & Rubin 1983 (*Biometrika*), *The central role of the propensity score in observational studies for causal effects*.

---
## 3 · Instrumental variables

> 📍 **Where we are.** Adjustment needs every confounder measured. An **instrument** offers a way around unmeasured
> confounding: a variable that nudges the exposure, is "as good as random", and affects the outcome only through the
> exposure. [Module 35](35_statistical_genetics.ipynb), Section 6, used genetic variants this way (Mendelian
> randomisation); here we build the general estimator and its main failure mode.

Our example: does the expression $X$ of a gene in blood causally raise a clinical marker $Y$? An unmeasured factor $U$
(inflammation, diet) raises both. Three **cis-eQTL** variants $Z$ near the gene shift its expression. If the variants
affect $Y$ only through $X$, they are valid instruments.

🧮 **Math refresher — the Wald ratio and two-stage least squares (2SLS).** With $X = \gamma Z + U + \varepsilon_X$ and
$Y = \theta X + U + \varepsilon_Y$, and $Z$ independent of $U$:

$$\operatorname{Cov}(Z, Y) = \theta \operatorname{Cov}(Z, X) \quad\Rightarrow\quad \hat\theta_{\text{Wald}} = \frac{\widehat{\operatorname{Cov}}(Z, Y)}{\widehat{\operatorname{Cov}}(Z, X)} .$$

*In words:* the instrument's effect on the outcome, divided by its effect on the exposure; $U$ drops out because $Z$ is
independent of it. With several instruments, **2SLS** first regresses $X$ on $Z$ (stage 1), then regresses $Y$ on the
*fitted* $\hat X$ (stage 2): only the part of $X$ driven by the instruments — free of $U$ — is used.

**The first-stage F-statistic** measures instrument strength (the F-test of stage 1). With weak instruments
($F$ below about 10 is the classic rule of thumb, Staiger & Stock 1997), 2SLS is biased towards the confounded
least-squares estimate and its sampling distribution has very heavy tails.

> 🔗 **Builds on** [Module 06](06_linear_regression_gwas.ipynb) (least squares) and [Module 35](35_statistical_genetics.ipynb)
> (Wald ratio and IVW with GWAS summary statistics, pleiotropy-robust MR-Egger and weighted median — not repeated here).

In [ ]:
# ── 3.1 A SIMULATED eQTL instrument: OLS vs 2SLS, for strong and weak instruments ──
def simulate_iv_study(rng, n=1000, gamma=0.5, theta=0.5):
    """Z: 3 cis-eQTL genotypes (0/1/2); X: gene expression; Y: clinical marker; U: unmeasured confounder of X and Y."""
    Z = rng.binomial(2, 0.3, (n, 3)).astype(float)
    u = rng.normal(size=n)
    x = Z @ np.full(3, gamma) + u + rng.normal(size=n)
    y = theta * x + u + rng.normal(size=n)
    return Z, x, y


def ols_slope(x, y):
    return np.polyfit(x, y, 1)[0]


def two_stage_least_squares(Z, x, y):
    """2SLS estimate of the effect of x on y, and the first-stage F-statistic."""
    Z1 = np.column_stack([np.ones(len(x)), Z])
    coef, rss1 = np.linalg.lstsq(Z1, x, rcond=None)[:2]
    x_hat = Z1 @ coef
    rss0 = np.sum((x - x.mean()) ** 2)
    F = ((rss0 - rss1[0]) / Z.shape[1]) / (rss1[0] / (len(x) - Z1.shape[1]))
    return ols_slope(x_hat, y), F


rng = np.random.default_rng(SEED + 6)
rows, draws = {}, {}
for gamma in [0.5, 0.2, 0.06]:
    res = np.array([[ols_slope(x_, y_), *two_stage_least_squares(Z_, x_, y_)] for Z_, x_, y_ in
                    (simulate_iv_study(rng, gamma=gamma) for _ in range(500))])
    draws[gamma] = res
    rows[f"γ = {gamma}"] = {"median first-stage F": np.median(res[:, 2]), "OLS: median": np.median(res[:, 0]),
                            "2SLS: median": np.median(res[:, 1]),
                            "2SLS: within ±0.25 of truth": np.mean(np.abs(res[:, 1] - 0.5) < 0.25)}
print("🧪 500 SIMULATED studies per instrument strength (n = 1000, true θ = 0.5):")
print(pd.DataFrame(rows).T.round(3).to_string())

In [ ]:
# ── 3.2 The instrument graph and the sampling distributions ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.6]})
draw_graph([("Z", "X"), ("X", "Y"), ("U", "X"), ("U", "Y")], {"Z": (0.05, 0.2), "X": (0.45, 0.2), "Y": (0.9, 0.2), "U": (0.67, 0.85)},
           ax=axes[0], highlight={"U"}, radius=0.08, edge_labels=["γ", "θ", "", ""],
           title="Z = eQTL genotypes, X = expression,\nY = marker, U = unmeasured confounder")
labels_ = [f"γ = {g}\n(F ≈ {np.median(r[:, 2]):.0f})" for g, r in draws.items()]
axes[1].boxplot([r[:, 1] for r in draws.values()], showfliers=False)
axes[1].set_xticks([1, 2, 3]); axes[1].set_xticklabels(labels_, fontsize=8)
axes[1].axhline(0.5, color="k", ls="--", lw=1, label="true θ = 0.5")
axes[1].axhline(np.median(draws[0.5][:, 0]), color=CLASS_COLORS[1], ls=":", lw=1.5, label="OLS (confounded)")
axes[1].set_ylabel("2SLS estimate of θ"); axes[1].legend(fontsize=7)
axes[1].set_title("SIMULATED: weaker instruments → wider and pulled towards OLS")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Least squares is biased in every scenario (median 0.93–1.0 instead of 0.5): $U$ raises both
expression and the marker, and OLS attributes that to expression. With strong instruments (median F ≈ 53) 2SLS is
centred on the truth (median 0.51) and every one of the 500 studies lands within ±0.25 of it. With moderate instruments
(F ≈ 9, just below the rule-of-thumb 10) it is roughly centred (0.54) but noisier: 81 % within ±0.25. With weak
instruments (F ≈ 1.3) the median drifts towards the OLS value (0.77) and only a third of the studies are within ±0.25
— and the extreme estimates (cut from the box plot) are wild: a ratio with a denominator close to zero.

**Instruments in experiments: compliance.** An instrument need not be genetic. In a randomised trial with imperfect
adherence, the *assignment* is an instrument for the treatment actually *taken*. Angrist, Imbens & Rubin (1996) showed
that with a binary instrument the Wald ratio — the intention-to-treat effect divided by the effect of assignment on
treatment uptake — estimates the **local average treatment effect** (LATE): the effect among the "compliers" whose
treatment follows their assignment. We will meet this exactly in Perturb-seq, where receiving a guide RNA is the
random assignment and some cells "do not comply" because the knockdown fails (Section 5).

⚠️ **The assumptions are untestable in the data.** Independence of $Z$ and $U$ and the exclusion restriction (no path
$Z \to Y$ except through $X$) must be argued from biology: a cis-eQTL might also alter a neighbouring gene (pleiotropy),
and population structure can correlate genotypes with environment ([Module 35](35_statistical_genetics.ipynb)).

✏️ **Your turn.** Add a direct effect of the first variant on the outcome (`y = theta * x + u + 0.2 * Z[:, 0] + rng.normal(size=n)`)
and re-run 3.1 for γ = 0.5. How large is the bias of 2SLS, and in which direction?

---
## 4 · Pooled CRISPR screens

> 📍 **Where we are.** From observational data to experiments: CRISPR lets us *set* a gene to "off" in every cell of a
> population — randomised intervention at genome scale.

### 🔬 Assay box — pooled CRISPR knockout screens

- **How it works.** A library of single-guide RNAs (sgRNAs) — typically 4–10 per gene for ~20,000 genes plus hundreds of
  non-targeting controls — is delivered by lentivirus at low multiplicity, so that most cells receive one guide. Cas9
  cuts the targeted gene and repair often disrupts it. The pool grows for one to three weeks; cells whose knockout slows
  growth become rarer. Guide sequences, integrated in the genome, are PCR-amplified and sequenced before (T0) and after
  selection; **guide counts** measure each knockout's fitness (Shalem *et al.* 2014; Wang *et al.* 2014). CRISPR
  interference (CRISPRi) and activation (CRISPRa) use a nuclease-dead Cas9 to repress or activate instead.
- **Typical sizes.** 50,000–100,000 guides, hundreds of cells per guide at every step (coverage), tens of millions of
  reads per sample.
- **Typical artefacts.** Guides differ in efficiency (some barely work); the library is unevenly represented at T0;
  bottlenecks (too few cells per guide) add random drift; low counts give noisy ratios; some guides deplete cells
  regardless of their target — e.g. by cutting in amplified genomic regions (Aguirre *et al.* 2016).

Our simulator encodes those ingredients, with 10 % essential genes whose loss reduces growth.

In [ ]:
# ── 4.1 A SIMULATED pooled knockout screen ──
def simulate_knockout_screen(rng, n_genes=1000, guides_per_gene=4, n_control_guides=200, frac_essential=0.1,
                             coverage=100, bottleneck=50, doublings=10, rogue_rate=0.03):
    """Guide counts before (T0) and after (T1) growth. Essential genes lose 0.03–0.3 log2 units of growth per doubling
    (scaled by each guide's efficiency); a few 'rogue' guides deplete cells whatever their target."""
    n_guides = n_genes * guides_per_gene + n_control_guides
    gene = np.r_[np.repeat(np.arange(n_genes), guides_per_gene), -np.ones(n_control_guides, int)]
    essential = rng.random(n_genes) < frac_essential
    fitness = np.where(essential, -rng.uniform(0.03, 0.3, n_genes), 0.0)
    efficiency = np.where(gene >= 0, rng.beta(5, 2, n_guides), 0.0)
    efficiency[rng.random(n_guides) < 0.1] *= 0.1                          # ~10 % of guides barely work
    rogue = (gene >= 0) & (rng.random(n_guides) < rogue_rate)
    log2_rate = np.where(gene >= 0, fitness[np.maximum(gene, 0)] * efficiency, 0.0) - rogue * rng.uniform(0.1, 0.3, n_guides)
    library = np.exp(rng.normal(0, 0.6, n_guides)); library /= library.sum()          # uneven library
    cells0 = rng.multinomial(n_guides * bottleneck, library)                          # cells per guide at T0
    grown = cells0 * 2.0 ** (doublings * (1 + log2_rate))
    cells1 = rng.multinomial(n_guides * bottleneck, grown / grown.sum())              # passage bottleneck
    reads0 = rng.multinomial(n_guides * coverage, cells0 / cells0.sum())              # sequencing
    reads1 = rng.multinomial(n_guides * coverage, cells1 / cells1.sum())
    guides = pd.DataFrame({"gene": gene, "efficiency": efficiency, "rogue": rogue, "T0": reads0, "T1": reads1})
    return guides, essential, fitness


guides, essential, fitness = simulate_knockout_screen(np.random.default_rng(SEED + 7))
targeting = guides.gene.to_numpy() >= 0
print(f"SIMULATED screen: {targeting.sum()} targeting + {(~targeting).sum()} non-targeting guides; "
      f"{essential.sum()} truly essential genes of {len(essential)}")
print(f"reads per guide at T0: median {guides.T0.median():.0f}, 5th–95th percentile {guides.T0.quantile(0.05):.0f}–{guides.T0.quantile(0.95):.0f}; "
      f"rogue guides: {guides.rogue.sum()}")

**Guide log fold changes.** For guide $g$ with counts $c_{g,0}$ and $c_{g,1}$ in libraries of $N_0$ and $N_1$ reads,

$$\text{LFC}_g = \log_2\frac{c_{g,1} + 0.5}{N_1} - \log_2\frac{c_{g,0} + 0.5}{N_0} - \operatorname{median}_{\text{controls}}(\cdot) .$$

*In words:* the change in each guide's share of the library, with a pseudo-count against zeros, centred so that the
non-targeting controls (which should not change) sit at 0.

**Gene-level aggregation.** Four guides per gene give four noisy, partly unreliable votes. Simple summaries are the mean
or median LFC. **Robust rank aggregation** (RRA; Kolde *et al.* 2012), used by MAGeCK (Li *et al.* 2014), asks whether
a gene's guides rank *unusually high* among all guides: rank every guide by LFC (most depleted first) and convert ranks
to $u \in (0, 1]$. If a gene's $k$ guides were random, their sorted ranks $u_{(1)} \le \dots \le u_{(k)}$ would be the order
statistics of $k$ uniform variables, and the $j$-th smallest follows a $\text{Beta}(j, k - j + 1)$ distribution. The score

$$\rho = \min_{j}\; P\big(\text{Beta}(j, k - j + 1) \le u_{(j)}\big)$$

*In words:* for each $j$, how surprising it is that the gene's $j$-th best guide ranks this high, keeping the most
surprising — so three strong guides make a gene significant even if the fourth failed. We turn $\rho$ into p-values by
permutation (random sets of four guides) and control the FDR with Benjamini–Hochberg. (MAGeCK also models guide counts
with a negative binomial — [Module 24](24_glm_count_data.ipynb) — to weigh low-count guides properly.)

In [ ]:
# ── 4.2 Guide LFCs, gene-level scores and 🧪 precision/recall for the essential genes ──
def guide_lfc(guides, pseudo=0.5):
    lfc = np.log2((guides.T1 + pseudo) / guides.T1.sum()) - np.log2((guides.T0 + pseudo) / guides.T0.sum())
    return (lfc - np.median(lfc[guides.gene < 0])).to_numpy()


def rra_rho(u_sorted):
    """RRA score per gene from its guides' sorted normalised ranks (genes × k)."""
    k = u_sorted.shape[1]
    j = np.arange(1, k + 1)
    return stats.beta.cdf(u_sorted, j, k - j + 1).min(axis=1)


lfc = guide_lfc(guides)
gene_of = guides.gene.to_numpy()[targeting]
u = stats.rankdata(lfc[targeting]) / targeting.sum()                      # 0 = most depleted guide
u_by_gene = np.sort(u.reshape(-1, 4), axis=1)                              # guides are stored gene by gene
rho = rra_rho(u_by_gene)
rng = np.random.default_rng(SEED + 8)
null = np.sort(np.concatenate([rra_rho(np.sort(rng.permutation(u).reshape(-1, 4), axis=1)) for _ in range(50)]))
q_rra = benjamini_hochberg((np.searchsorted(null, rho, side="right") + 1) / (len(null) + 1))
gene_mean = lfc[targeting].reshape(-1, 4).mean(axis=1)
gene_median = np.median(lfc[targeting].reshape(-1, 4), axis=1)
ctrl_lfc = lfc[~targeting]                                                 # naive z-test of the mean against controls
q_mean = benjamini_hochberg(stats.norm.cdf((gene_mean - ctrl_lfc.mean()) / (ctrl_lfc.std() / 2)))

rows = {}
for name, score, q in [("mean LFC (z-test vs controls)", -gene_mean, q_mean), ("median LFC", -gene_median, None),
                       ("RRA (permutation p-values)", -np.log(rho), q_rra)]:
    row = {"AUPRC": average_precision_score(essential, score)}
    if q is not None:
        p, r, _ = precision_recall_f1(essential.astype(int), (q < 0.1).astype(int))
        row.update({"calls at FDR 10 %": int((q < 0.1).sum()), "precision": p, "recall": r})
    rows[name] = row
print(f"🧪 {essential.sum()} truly essential genes:")
print(pd.DataFrame(rows).T.round(3).to_string(na_rep="—"))

In [ ]:
# ── 4.3 What the screen looks like ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
ess_guide = np.r_[np.repeat(essential, 4), np.zeros((~targeting).sum(), bool)]
bins = np.linspace(-3.5, 1.5, 60)
for mask, name, col in [(targeting & ~ess_guide, "non-essential genes", "0.6"), (ess_guide, "essential genes", CLASS_COLORS[1]),
                        (~targeting, "non-targeting controls", CLASS_COLORS[0])]:
    axes[0].hist(lfc[mask], bins=bins, density=True, histtype="step", lw=1.8, color=col, label=name)
axes[0].set_xlabel("guide LFC (log2)"); axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
axes[0].set_title("essential genes' guides are depleted")
axes[1].scatter(np.log10(guides.T0[~targeting]), lfc[~targeting], s=6, color=CLASS_COLORS[0])
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel("log10 T0 reads (non-targeting guides)"); axes[1].set_ylabel("LFC")
axes[1].set_title("pure noise: larger at low counts")
for k, (name, s) in enumerate([("mean LFC", -gene_mean), ("median LFC", -gene_median), ("RRA", -np.log(rho))]):
    prec, rec, _ = precision_recall_curve(essential, s)
    axes[2].plot(rec, prec, color=CLASS_COLORS[k], label=f"{name} (AUPRC {average_precision_score(essential, s):.2f})")
axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].legend(fontsize=7)
axes[2].set_title("ranking the essential genes")
plt.suptitle("SIMULATED pooled CRISPR knockout screen", y=1.02)
plt.tight_layout(); plt.show()
c0 = guides.T0.to_numpy()[~targeting]
cuts = np.quantile(c0, [1 / 3, 2 / 3])
print("SD of non-targeting LFCs by T0-count tercile (low → high):",
      [round(float(ctrl_lfc[np.digitize(c0, cuts) == k].std()), 2) for k in range(3)])
found = essential & (q_rra < 0.1)
print(f"growth defect per doubling (median) — essential genes found by RRA: {np.median(-fitness[found]):.2f}; "
      f"missed: {np.median(-fitness[essential & ~found]):.2f}")

🔍 **What just happened?** The guides of essential genes are shifted to negative LFCs, but the distributions overlap:
weakly essential genes, inefficient guides and count noise blur the picture. The non-targeting controls show the noise
alone — and it is larger for guides with few T0 reads (SD 0.48 in the lowest third of counts, 0.22 in the highest), the
reason for pseudo-counts and count models.

The gene-level table shows two lessons. First, **aggregation should be robust**: the median and RRA rank essential genes
better than the mean (AUPRC 0.93 and 0.91 vs 0.84), because a single rogue guide — depleting cells whatever its target —
drags a non-essential gene's mean down. Second, **the test matters as much as the score**: the naive z-test of the mean
against the control guides calls 168 genes at FDR 10 %, but its realised precision is only 0.55 — it ignores the rogue
guides and treats the guide noise as known — whereas RRA with permutation p-values keeps its promise (61 calls, all
correct) at the cost of recall (0.59). The essential genes RRA misses are mostly the weakest ones (median growth defect
0.11 log2 units per doubling, against 0.21 for those it finds), too small to see after ten doublings with this much
noise.

✏️ **Your turn.** Re-run the simulator with `coverage=500, bottleneck=300`. Which numbers improve most — and what does
that say about where the noise in this screen comes from?

---
## 5 · Perturb-seq

> 📍 **Where we are.** A pooled screen measures one number per knockout (fitness). Perturb-seq measures a whole
> transcriptome per perturbed cell — so we can ask *which genes* each knockdown changes, and from there, how genes are
> wired.

### 🔬 Assay box — Perturb-seq

- **How it works.** A pooled CRISPR library (usually CRISPRi, which represses transcription) is delivered to cells as
  in Section 4, but instead of counting guides after selection, cells go into droplet scRNA-seq. Each guide is expressed
  with a capturable barcode (or captured directly, as in CROP-seq), so every cell's transcriptome comes with the identity
  of its perturbation (Dixit *et al.* 2016; Adamson *et al.* 2016). Several guides per cell give combinatorial
  perturbations (Norman *et al.* 2019).
- **Typical sizes.** From tens of perturbations to genome scale (Replogle *et al.* 2022: millions of cells), typically
  ~100 cells per perturbation and thousands of control cells with non-targeting guides.
- **Typical artefacts.** **Escape** — some cells carry the guide but keep a working gene (inefficient repression,
  silencing of the guide); guide assignment errors and cells with two guides; knockdown efficiency differs between
  guides; perturbations change cell-cycle and cell state, and strong knockdowns change cell size and total RNA content.

The toolkit's `simulate_perturb_seq` builds a random DAG of 40 genes with signed edges. A cell's deviation from
baseline propagates down the DAG with a **saturating** response, $d_g = c\,\tanh\big((\sum_{\text{parents}} w\,d_{\text{parent}} + \varepsilon_g)/c\big)$,
and a knockdown *clamps* its target at $-2$ — an intervention $do(X_{\text{target}} = -2)$, i.e. graph surgery. About 10 %
of the cells of each guide escape. We request 15 single knockdowns and 40 double knockdowns.

In [ ]:
# ── 5.1 The SIMULATED Perturb-seq screen and its hidden network ──
ps = simulate_perturb_seq(np.random.default_rng(SEED + 1), n_genes=40, n_targets=15, n_pairs=40,
                          cells_per_perturbation=100, n_control=1000)
counts, latent, pobs, pvar = ps["counts"], ps["expression"], ps["obs"], ps["var"]
W_true, true_effect = ps["W"], ps["true_effect"]
genes = pvar.index.to_numpy(str)
gidx = {g: i for i, g in enumerate(genes)}
label = pobs.perturbation.to_numpy(str)
is_ctrl = label == "ctrl"
targets = list(genes[pvar.is_target.to_numpy(bool)])
singles = [p for p in true_effect.index if p != "ctrl" and "+" not in p]
doubles = [p for p in true_effect.index if "+" in p]
A_true = (W_true.to_numpy() != 0).astype(int)
desc_true = A_true.copy()                                     # transitive closure: who is downstream of whom
for _ in range(len(genes)):
    desc_true = ((desc_true + desc_true @ A_true) > 0).astype(int)
print(f"SIMULATED Perturb-seq: {counts.shape[0]} cells × {counts.shape[1]} genes; {is_ctrl.sum()} control cells, "
      f"{len(singles)} single and {len(doubles)} double knockdowns; median UMIs per cell {np.median(counts.sum(axis=1)):.0f}")
print(f"hidden DAG: {A_true.sum()} direct edges; the 15 targets have {desc_true[[gidx[t] for t in targets]].sum()} "
      f"downstream (descendant) genes in total")

order = np.argsort(pvar.topological_rank.to_numpy())
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(W_true.to_numpy()[np.ix_(order, order)], cmap="RdBu_r", vmin=-1.2, vmax=1.2)
ax.set_xlabel("child (topological order)"); ax.set_ylabel("parent (topological order)")
tpos = [int(np.flatnonzero(order == gidx[t])[0]) for t in targets]
ax.scatter([-1.2] * len(tpos), tpos, marker=">", color="k", s=12, clip_on=False, label="knocked-down genes")
plt.colorbar(im, ax=ax, shrink=0.8, label="edge weight w"); ax.legend(fontsize=7, loc="lower left")
ax.set_title("SIMULATED true network: a DAG is upper-triangular\nwhen genes are sorted in causal order")
plt.tight_layout(); plt.show()

**Normalisation, revisited.** Module 30's total-count normalisation divides each cell by its total UMIs. With only 40
genes, a knockdown of an abundant gene noticeably shrinks the total, so after normalisation *every other gene* appears
to go up — a compositional artefact (real screens have thousands of genes, which makes it smaller but not zero).
[Module 24](24_glm_count_data.ipynb)'s **median-of-ratios** size factor is robust to it: it assumes most genes do not
change, which holds for most perturbations. Let us compare both with the truth, using the genes that a knockdown truly
does *not* affect.

In [ ]:
# ── 5.2 🧪 Two normalisations against the true effects of the 15 single knockdowns ──
def median_of_ratios_log(C, pseudo=0.5):
    """log counts minus a per-cell median-of-ratios size factor (Module 24), robust to a few changing genes."""
    lc = np.log(C + pseudo)
    return lc - np.median(lc - lc.mean(axis=0), axis=1, keepdims=True)


def mean_shift(X, cells):
    """Average expression change of a group of cells relative to the control cells."""
    return X[cells].mean(axis=0) - X[is_ctrl].mean(axis=0)


X_total, X_mor = log_normalize(counts), median_of_ratios_log(counts)
rows = {}
for name, X in [("total-count (log_normalize)", X_total), ("median-of-ratios", X_mor)]:
    shift_unaff, err = [], []
    for p in singles:
        est_p, tru = mean_shift(X, label == p), true_effect.loc[p].to_numpy()
        shift_unaff.append(np.median(est_p[tru == 0]))
        err.append(np.mean(np.abs(est_p - tru)))
    rows[name] = {"median shift of unaffected genes (mean |·|)": np.mean(np.abs(shift_unaff)),
                  "largest |shift|": np.max(np.abs(shift_unaff)), "MAE vs true effect": np.mean(err)}
print(pd.DataFrame(rows).T.round(3).to_string())
X = X_mor                                                      # used from here on

🔍 **What just happened?** With total-count normalisation, the genes that a knockdown does not touch appear shifted by
about 0.09 on average (up to 0.17) — up or down, depending on whether the knockdown lowered or raised the cell's total
RNA. Median-of-ratios brings that to 0.02 (at most 0.06) and cuts the error against the true effects from 0.11 to 0.07.
We use it from now on.

### Guide escape: a mixture model per perturbation

About one cell in ten carrying a guide has escaped the knockdown. Averaging over all guide-carrying cells (the
intention-to-treat effect) therefore dilutes every effect. **Mixscape** (Papalexi *et al.* 2021) classifies cells into
perturbed and escaped. Our simplified version, per perturbation:

1. compute the perturbation **signature** $v$: the mean expression difference between the guide's cells and the controls;
2. project every guide cell and every control cell onto $v$ — a one-number summary of "how perturbed" a cell looks;
3. fit a **two-component Gaussian mixture** by EM to the guide cells' projections, with one component *fixed* to the
   control distribution (escaped cells look like controls) and the other free;
4. call cells perturbed if their posterior probability exceeds 0.5; recompute $v$ from them and repeat twice.

> 🔗 **Builds on** [Module 13](13_mixture_models_em.ipynb): the E-step computes responsibilities, the M-step updates the
> free component's mean, SD and the mixing weight; fixing one component's parameters is a constraint that EM handles
> by simply not updating them.

The hidden truth: in the simulator's latent expression, a perturbed cell has its target *exactly* at baseline − 2,
an escaped cell does not.

In [ ]:
# ── 5.3 Mixscape-lite: escape detection with a constrained two-component mixture ──
def em_fixed_null(s, m0, s0, n_iter=200):
    """EM for a 1-D mixture of a FIXED 'escaped' Gaussian N(m0, s0²) and a free 'perturbed' Gaussian."""
    m1, s1, pi = (np.quantile(s, 0.1) if np.quantile(s, 0.1) < m0 else np.quantile(s, 0.9)), s0, 0.5
    for _ in range(n_iter):
        d0, d1 = (1 - pi) * stats.norm.pdf(s, m0, s0), pi * stats.norm.pdf(s, m1, s1)
        r = d1 / (d0 + d1 + 1e-300)                                   # E-step: P(perturbed | s)
        pi, m1 = r.mean(), (r * s).sum() / r.sum()                    # M-step for the free component only
        s1 = max(np.sqrt((r * (s - m1) ** 2).sum() / r.sum()), 0.05)
    return r, (m1, s1, pi)


def mixscape_lite(X, p, n_rounds=3):
    """Classify the cells of perturbation p as perturbed (True) or escaped (False)."""
    cells, mu0 = np.flatnonzero(label == p), X[is_ctrl].mean(axis=0)
    ko = np.ones(len(cells), bool)
    for _ in range(n_rounds):
        v = X[cells][ko].mean(axis=0) - mu0
        v /= np.linalg.norm(v)
        s_guide, s_ctrl = (X[cells] - mu0) @ v, (X[is_ctrl] - mu0) @ v
        r, fit = em_fixed_null(s_guide, s_ctrl.mean(), s_ctrl.std())
        ko = r > 0.5
    return ko, s_guide, s_ctrl, fit


def truly_perturbed(p):
    """Hidden truth: every target of p sits exactly at baseline − 2 in the latent expression."""
    cells = label == p
    return np.all([np.isclose(latent[cells, gidx[t]] - pvar.baseline[t], -2.0) for t in p.split("+")], axis=0)

In [ ]:
# ── 5.4 🧪 Escape detection against the hidden truth ──
ko_call, rows = {}, []
for p in singles:
    ko_call[p] = mixscape_lite(X, p)[0]
    tru = truly_perturbed(p)
    rows.append({"perturbation": p, "escaped (truth)": int((~tru).sum()), "called escaped": int((~ko_call[p]).sum()),
                 "accuracy": np.mean(ko_call[p] == tru), "escape recall": np.mean(~ko_call[p][~tru]),
                 "escape precision": np.mean(~tru[~ko_call[p]]) if (~ko_call[p]).any() else np.nan})
esc = pd.DataFrame(rows).set_index("perturbation")
print("🧪 escape detection, 15 single knockdowns:")
print(esc.round(2).to_string())
print(f"\noverall: accuracy {esc.accuracy.mean():.3f}; escaped cells found {esc['escape recall'].mean():.2f}; "
      f"calls that are truly escaped {esc['escape precision'].mean():.2f}")

In [ ]:
# ── 5.5 One perturbation up close ──
p_show = esc["escaped (truth)"].idxmax()
ko, s_guide, s_ctrl, (m1, s1, pi1) = mixscape_lite(X, p_show)
tru = truly_perturbed(p_show)
fig, ax = plt.subplots(figsize=(7.5, 3.2))
bins = np.linspace(min(s_guide.min(), s_ctrl.min()), max(s_guide.max(), s_ctrl.max()), 45)
ax.hist(s_ctrl, bins=bins, density=True, color="0.8", label="control cells")
ax.hist(s_guide[tru], bins=bins, density=True, alpha=0.6, color=CLASS_COLORS[0], label="guide cells: perturbed (truth)")
ax.hist(s_guide[~tru], bins=bins, density=True, alpha=0.8, color=CLASS_COLORS[1], label="guide cells: escaped (truth)")
grid = np.linspace(bins[0], bins[-1], 200)
ax.plot(grid, stats.norm.pdf(grid, m1, s1), color=CLASS_COLORS[0], lw=1.5, ls="--", label="fitted perturbed component")
ax.plot(grid, stats.norm.pdf(grid, s_ctrl.mean(), s_ctrl.std()), color="0.3", lw=1.5, ls="--", label="fixed escaped component")
ax.set_xlabel("projection on the perturbation signature"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title(f"SIMULATED knockdown of {p_show}: escaped cells look like controls")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The classifier recovers nearly every cell's status: overall accuracy is about 0.99, it finds
most escaped cells and almost all its "escaped" calls are right. The histogram shows why it works: escaped cells
(red) sit exactly on the control distribution, perturbed cells form a separate bump. The signature uses *all* genes,
not only the target — useful because a weakly expressed target gives too few UMIs to tell knockdown from noise, while
its downstream genes still testify. (On real data the separation is much less clean; Mixscape also first removes
cell-state variation by comparing each cell with its nearest control neighbours.)

**Three ways to estimate a knockdown's effect.** (1) The **intention-to-treat** mean over all guide cells; (2) the mean
over the cells called perturbed; (3) the **instrumental-variable** view of Section 3: guide assignment is the
instrument, the drop in target expression is the "treatment taken", so dividing the ITT shift of every gene by the ITT
shift of the target and multiplying by the full knockdown gives a Wald-type estimate of the effect among compliers.

In [ ]:
# ── 5.6 🧪 Effect estimates against the true effects ──
rows = {"intention-to-treat (all guide cells)": [], "perturbed cells only (Mixscape-lite)": [], "IV / Wald (ITT ÷ compliance)": []}
for p in singles:
    tru, j = true_effect.loc[p].to_numpy(), gidx[p]
    itt = mean_shift(X, label == p)
    ko_cells = np.flatnonzero(label == p)[ko_call[p]]
    ko_eff = mean_shift(X, ko_cells)
    wald = itt * ko_eff[j] / itt[j]                     # rescale by (full knockdown ÷ ITT knockdown of the target)
    for name, est_p in zip(rows, [itt, ko_eff, wald]):
        rows[name].append(np.mean(np.abs(est_p - tru)[np.arange(len(genes)) != j]))
print("🧪 mean absolute error over non-target genes, averaged over the 15 single knockdowns:")
print({k: round(float(np.mean(v)), 4) for k, v in rows.items()})

🔍 **What just happened?** All three estimates are close to the truth (mean absolute errors 0.058–0.066), and the two that
account for escape beat the diluted intention-to-treat average (0.066). The Wald rescaling recovers nearly the same
accuracy as explicit classification, without classifying a single cell — Section 3's instrumental-variable logic
applied to CRISPR. (Our rescaling borrows the full-knockdown size of the target from the perturbed cells; in practice it
is often known from the design or from target-gene measurements.)

✏️ **Your turn.** Simulate a screen with `escape_rate=0.4`. How do the intention-to-treat errors change, and does
Mixscape-lite still find the escaped cells?

### 📄 Research connection — CRISPR screens and Perturb-seq

- **What the field does.** Genome-scale pooled knockout screens (Shalem *et al.* 2014; Wang *et al.* 2014) mapped essential genes in human
    cell lines; MAGeCK (Li *et al.* 2014) and BAGEL are standard analysis tools, and large efforts such as DepMap screen
    hundreds of cancer cell lines. Perturb-seq / CRISP-seq / CROP-seq (Dixit *et al.* 2016; Adamson *et al.* 2016; Jaitin
    *et al.* 2016; Datlinger *et al.* 2017) added single-cell transcriptomes; Replogle *et al.* (2022) reached genome
    scale; Mixscape (Papalexi *et al.* 2021) handles escaping cells.
- **What this notebook builds.** A pooled screen simulator with guide efficiency, rogue guides, library skew and bottlenecks, analysed with
    LFCs and RRA and graded against true essential genes; Perturb-seq effects with a robust normalisation, a constrained
    mixture model for escape, and an IV view of guide assignment, all graded against the true effects.
- **Where the frontier is.** Readouts are expanding beyond RNA (protein, chromatin, imaging-based optical pooled screens), perturbations to
    base and prime editing of variants, and scale to in-vivo screens in tissues. Analysis challenges include
    cell-state heterogeneity, guide-level variability, and statistically efficient designs with fewer cells per
    perturbation or compressed (pooled) perturbations.
- **Read next.** Replogle *et al.* 2022 (*Cell*), *Mapping information-rich genotype–phenotype landscapes with genome-scale Perturb-seq*; Dixit *et al.* 2016 (*Cell*), *Perturb-Seq*; Li *et al.* 2014 (*Genome Biology*), *MAGeCK*.

---
## 6 · Learning the network

> 📍 **Where we are.** Which gene regulates which? [Module 17](17_bayesian_networks_inference_learning.ipynb), Section 10,
> ended with the problem: observational data identify a network only up to **Markov equivalence** — the skeleton and
> the v-structures, not the direction of most edges. Interventions break the symmetry: knocking down a gene changes
> its **descendants** and nothing upstream.

> 🔗 **Builds on** [Module 18](18_gaussian_graphical_models.ipynb): in a Gaussian graphical model, a zero partial
> correlation (a zero in the precision matrix) means conditional independence. For data generated by a DAG, the
> GGM's graph is the **moral graph**: every true edge plus an edge between every pair of co-parents of a common child.

**Step 1 — observational cells only.** Estimate partial correlations from the 1,000 control cells (inverse of the
covariance matrix) and keep the edges with a Fisher-z test significant at a strict FDR (q < 0.001; with 1,000 cells even
tiny artefactual partial correlations become "significant"). This gives an undirected skeleton.

In [ ]:
# ── 6.1 Observational skeleton from the control cells (partial correlations) ──
n_g = len(genes)
iu = np.triu_indices(n_g, 1)
Xc = X[is_ctrl]
P = np.linalg.inv(np.cov(Xc.T))
pcor = -P / np.sqrt(np.outer(np.diag(P), np.diag(P)))
zstat = np.arctanh(np.clip(pcor, -0.999, 0.999)) * np.sqrt(len(Xc) - (n_g - 2) - 3)
q = benjamini_hochberg(2 * stats.norm.sf(np.abs(zstat[iu])))
skeleton = np.zeros((n_g, n_g), bool)
skeleton[iu] = q < 0.001
skeleton |= skeleton.T

true_skel = (A_true + A_true.T) > 0
moral = true_skel.copy()
for child in range(n_g):                                           # marry the parents of every child
    pa = np.flatnonzero(A_true[:, child])
    moral[np.ix_(pa, pa)] = True
np.fill_diagonal(moral, False)
for name, ref in [("true skeleton", true_skel), ("moral graph", moral)]:
    p_, r_, _ = precision_recall_f1(ref[iu].astype(int), skeleton[iu].astype(int))
    print(f"🧪 estimated skeleton ({skeleton[iu].sum()} edges) vs {name} ({ref[iu].sum()} edges): precision {p_:.2f}, recall {r_:.2f}")

🔍 **What just happened?** The skeleton recovers most true regulatory edges (recall 0.84), but only 56 % of its 88 edges
are true edges. Measured against the moral graph its precision is 0.72, so about 16 % of the edges connect co-parents
of a common child — exactly what a GGM should find — and the remaining ~28 % are artefacts of counting noise and
normalisation, which break the conditional independences a mediator should create.
And the skeleton has no arrows: among the edges touching a knocked-down gene, observational data alone cannot say which
way they point.

**Step 2 — interventions.** For every single knockdown, test every gene for a change between the cells called
perturbed and the controls (Welch t-test, BH at 5 %). A gene that changes is a **descendant** of the target. The set of
descendants is the transitive closure of the true graph from that node — direct and indirect targets together.

**Step 3 — from descendants to direct edges.** Two ideas, which we also combine:

- **transitive reduction**: if knocking down A changes B and C, and knocking down B (also a target) changes C, then the
  effect of A on C may be indirect, via B — remove A → C;
- **intersection with the skeleton**: keep A → C only if A and C are adjacent in the observational skeleton — direction
  from the intervention, adjacency from conditional independence.

In [ ]:
# ── 6.2 Interventions: descendants, transitive reduction and 🧪 precision/recall against the true DAG ──
tidx = np.array([gidx[t] for t in targets])
desc_hat = np.zeros((n_g, n_g), bool)
for p in singles:
    cells = np.flatnonzero(label == p)[ko_call[p]]
    q_p = benjamini_hochberg(stats.ttest_ind(X[cells], X[is_ctrl], equal_var=False).pvalue)
    hit = q_p < 0.05
    hit[gidx[p]] = False
    desc_hat[gidx[p]] = hit
reduced = desc_hat.copy()
for a_ in tidx:
    for b_ in tidx:
        if a_ != b_ and desc_hat[a_, b_]:
            reduced[a_] &= ~desc_hat[b_]                 # a → c is explained by a → b → c

true_direct, true_desc = A_true.astype(bool)[tidx], desc_true.astype(bool)[tidx]
p_, r_, _ = precision_recall_f1(true_desc.ravel().astype(int), desc_hat[tidx].ravel().astype(int))
print(f"🧪 descendant calls: {desc_hat[tidx].sum()} (true descendants: {true_desc.sum()}) — precision {p_:.2f}, recall {r_:.2f}")
rows = {}
for name, M in [("observational skeleton (edges as outgoing)", skeleton), ("descendants = edges", desc_hat),
                ("transitive reduction", reduced), ("descendants ∩ skeleton", desc_hat & skeleton),
                ("transitive reduction ∩ skeleton", reduced & skeleton)]:
    p_, r_, f_ = precision_recall_f1(true_direct.ravel().astype(int), M[tidx].ravel().astype(int))
    rows[name] = {"edges called": int(M[tidx].sum()), "precision": p_, "recall": r_, "F1": f_}
direct_table = pd.DataFrame(rows).T
print(f"\n🧪 direct edges out of the 15 knocked-down genes (true: {true_direct.sum()}):")
print(direct_table.round(2).to_string())

In [ ]:
# ── 6.3 Precision and recall of the five network estimates ──
fig, ax = plt.subplots(figsize=(9, 3.2))
xpos = np.arange(len(direct_table))
ax.bar(xpos - 0.2, direct_table.precision, 0.4, color=CLASS_COLORS[0], label="precision")
ax.bar(xpos + 0.2, direct_table.recall, 0.4, color=CLASS_COLORS[2], label="recall")
ax.set_xticks(xpos); ax.set_xticklabels([n.replace(" (", "\n(").replace(" ∩ ", "\nAND ") for n in direct_table.index], fontsize=7)
ax.set_ylim(0, 1.2); ax.legend(fontsize=8, ncol=2, loc="upper left")
ax.set_title("SIMULATED: direct regulatory edges of the knocked-down genes")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The interventions find every descendant (recall 1.0); about one call in four is spurious:
small shifts in genes outside the descendant set — residues of normalisation and noise — reach significance, since
with 100 perturbed cells per knockdown the t-test detects shifts of a few hundredths. Turned into
edges, the descendant sets have perfect recall but low precision: most descendants are *indirect* targets. Each
refinement trades a little recall for a lot of precision:

- the **observational skeleton alone** (all edges touching a target treated as outgoing) has low precision — it cannot
  tell direction, and contains moral and artefactual edges;
- **transitive reduction** removes indirect effects that pass through other knocked-down genes — but not those through
  genes we never perturbed;
- **intersecting with the skeleton** removes indirect effects through *any* gene, because a non-adjacent pair is
  conditionally independent given the mediators;
- intersecting the descendant sets with the skeleton gives the best balance (precision 0.87, recall 0.90); adding the
  transitive reduction raises precision slightly (0.89) at some cost in recall (0.83).

The lesson generalises: **observational data give adjacencies, interventions give directions**, and the combination is
much stronger than either. Principled versions of this idea score whole DAGs on mixed observational and interventional
data (GIES, Hauser & Bühlmann 2012) or look for relations that stay invariant across interventions (invariant causal
prediction; Meinshausen *et al.* 2016 validated such methods on yeast gene-knockout data).

⚠️ **Assumptions we relied on.** No hidden common causes of the measured genes, no feedback loops (the simulator's
network is acyclic — real networks are not), interventions that hit only their target (no off-target effects), and
cells in one homogeneous state. Each failure creates edges that do not exist or hides ones that do.

✏️ **Your turn.** Change the descendant threshold from q < 0.05 to q < 0.001. Which estimates gain precision, and
which lose recall? Why might a strict threshold miss true direct edges with small weights?

---
## 7 · Predicting unseen perturbations

> 📍 **Where we are.** Twenty thousand genes give 200 million gene pairs — no experiment will test them all. Can we
> predict the effect of a **double** knockdown from the two singles, or the effect of knocking down a gene we never
> perturbed?

### Double knockdowns: the additive baseline

The simplest prediction for A + B is the **sum of the single effects**, $\hat\delta_{A+B} = \hat\delta_A + \hat\delta_B$ — what
we expect if the two perturbations act independently (on the log scale). Departures from additivity are **genetic
interactions** (epistasis): synergy, buffering, masking. Two mechanisms in our simulator make doubles non-additive:
the saturating $\tanh$ response, and **masking** — if A acts on a gene through B, knocking B down as well removes A's
indirect effect, so adding both effects double-counts it. For the two targets themselves we always predict the
knocked-down level (we know we clamped them).

We estimate the single effects from the perturbed cells (cells 5.3–5.4) and compare predictions with the **true** double effects.

In [ ]:
# ── 7.1 The additive baseline for the 40 double knockdowns ──
single_eff = {p: mean_shift(X, np.flatnonzero(label == p)[ko_call[p]]) for p in singles}


def additive_prediction(pair):
    a_, b_ = pair.split("+")
    pred = single_eff[a_] + single_eff[b_]
    pred[gidx[a_]], pred[gidx[b_]] = single_eff[a_][gidx[a_]], single_eff[b_][gidx[b_]]   # the clamped targets
    return pred


def mse(a_, b_):
    return float(np.mean((np.asarray(a_) - np.asarray(b_)) ** 2))


pts = []                                                         # (single A, single B, true double, affected by both?)
for pair in doubles:
    a_, b_ = pair.split("+")
    ta, tb, tab = (true_effect.loc[k].to_numpy() for k in (a_, b_, pair))
    for g in set(range(n_g)) - {gidx[a_], gidx[b_]}:
        pts.append((ta[g] + tb[g], tab[g], abs(ta[g]) > 0.05 and abs(tb[g]) > 0.05))
pts = np.array(pts)
both = pts[:, 2].astype(bool)
print(f"MSE vs the true double effect — additive (estimated singles): "
      f"{np.mean([mse(additive_prediction(p), true_effect.loc[p]) for p in doubles]):.4f}; "
      f"additive of the TRUE singles: {np.mean([mse(true_effect.loc[p.split('+')[0]] + true_effect.loc[p.split('+')[1]], true_effect.loc[p]) for p in doubles]):.4f}; "
      f"predicting no change: {np.mean([mse(0, true_effect.loc[p]) for p in doubles]):.4f}")
print(f"gene–pair combinations affected by BOTH single knockdowns: {both.sum()} of {len(both)}; "
      f"additive MSE on those {np.mean((pts[both, 0] - pts[both, 1]) ** 2):.3f}, on the rest {np.mean((pts[~both, 0] - pts[~both, 1]) ** 2):.1e}")

### A learned combination rule

Can a model learn how effects combine? We train a small neural network $f(a, b)$ that predicts a gene's double-knockdown
effect from its two single effects $a$ and $b$, shared across all genes (2 inputs → 16 → 16 → 1, tanh units), written
as a correction to the additive baseline, $\hat\delta = a + b + \tfrac12[g(a, b) + g(b, a)]$, so that it is symmetric and
starts from additivity. It is trained on the *measured* double effects of 32 pairs and tested on the 8 held-out pairs
(5-fold cross-validation over pairs, [Module 23](23_model_selection.ipynb)), always scored against the true effects.

> 🔗 **Builds on** [Module 22](22_neural_networks.ipynb): `nn.Module`, `optim.Adam`, a full-batch training loop.

In [ ]:
# ── 7.2 A learned combination model and its training data ──
class CombinationMLP(torch.nn.Module):
    """δ(a, b) = a + b + ½[g(a, b) + g(b, a)]: symmetric, and additive when g = 0."""
    def __init__(self, hidden=16):
        super().__init__()
        self.g = torch.nn.Sequential(torch.nn.Linear(2, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, hidden),
                                     torch.nn.Tanh(), torch.nn.Linear(hidden, 1))

    def forward(self, ab):
        return ab.sum(dim=1) + 0.5 * (self.g(ab) + self.g(ab.flip(1))).squeeze(1)


def pair_features(pair):
    """Single effects (a, b) of every non-target gene and the MEASURED double effect (perturbed cells)."""
    a_, b_ = pair.split("+")
    keep = np.setdiff1d(np.arange(n_g), [gidx[a_], gidx[b_]])
    measured = mean_shift(X, np.flatnonzero(label == pair)[mixscape_lite(X, pair)[0]])
    return np.column_stack([single_eff[a_], single_eff[b_]])[keep], measured[keep], keep


feats = {p: pair_features(p) for p in doubles}
print(f"{len(doubles)} pairs × {len(feats[doubles[0]][1])} non-target genes; parameters in the model: "
      f"{sum(w.numel() for w in CombinationMLP().parameters())}")

In [ ]:
# ── 7.3 🧪 Additive baseline vs the learned model, cross-validated over pairs ──
folds = np.array_split(np.random.default_rng(SEED + 9).permutation(len(doubles)), 5)
res = {"additive": [], "learned MLP": [], "measured double (perturbed cells)": []}
t0 = time.time()
for fold in folds:
    test_pairs = [doubles[i] for i in fold]
    train_pairs = [p for p in doubles if p not in test_pairs]
    Xtr = torch.tensor(np.vstack([feats[p][0] for p in train_pairs]), dtype=torch.float32)
    ytr = torch.tensor(np.concatenate([feats[p][1] for p in train_pairs]), dtype=torch.float32)
    torch.manual_seed(SEED)
    model = CombinationMLP()
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    for step in range(300):
        opt.zero_grad()
        loss = torch.mean((model(Xtr) - ytr) ** 2)
        loss.backward()
        opt.step()
    for p in test_pairs:
        ab, measured, keep = feats[p]
        with torch.no_grad():
            learned = additive_prediction(p)
            learned[keep] = model(torch.tensor(ab, dtype=torch.float32)).numpy()
        full_measured = additive_prediction(p)
        full_measured[keep] = measured
        for name, pred in zip(res, [additive_prediction(p), learned, full_measured]):
            res[name].append(mse(pred, true_effect.loc[p]))
better = np.mean(np.array(res["learned MLP"]) < np.array(res["additive"]))
print(f"🧪 mean squared error vs the TRUE double effects (40 held-out pairs, {time.time() - t0:.1f} s):")
print({k: round(float(np.mean(v)), 4) for k, v in res.items()})
print(f"the learned model beats the additive baseline on {better:.0%} of the pairs")

In [ ]:
# ── 7.4 Where additivity fails, and what the model learned ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(pts[~both, 0], pts[~both, 1], s=6, color="0.6", label="gene affected by one knockdown")
axes[0].scatter(pts[both, 0], pts[both, 1], s=14, color=CLASS_COLORS[1], label="gene affected by both")
lim = [pts[:, :2].min() - 0.2, pts[:, :2].max() + 0.2]
axes[0].plot(lim, lim, "k--", lw=0.8)
axes[0].set_xlabel("sum of the TRUE single effects"); axes[0].set_ylabel("TRUE double effect"); axes[0].legend(fontsize=7)
axes[0].set_title("SIMULATED: additivity holds unless both knockdowns act on a gene")
grid = np.linspace(-1.5, 1.5, 61)
G1, G2 = np.meshgrid(grid, grid)
with torch.no_grad():
    corr = model(torch.tensor(np.c_[G1.ravel(), G2.ravel()], dtype=torch.float32)).numpy() - (G1.ravel() + G2.ravel())
im = axes[1].contourf(G1, G2, corr.reshape(G1.shape), levels=np.linspace(-1, 1, 21), cmap="RdBu_r", extend="both")
plt.colorbar(im, ax=axes[1], label="learned − additive")
ab_all = np.vstack([feats[p][0] for p in doubles])
hit2 = (np.abs(ab_all) > 0.2).all(axis=1)
axes[1].scatter(ab_all[hit2, 0], ab_all[hit2, 1], s=8, color="k", label="training genes with |a|, |b| > 0.2")
axes[1].legend(fontsize=7, loc="lower right")
axes[1].set_xlabel("single effect a"); axes[1].set_ylabel("single effect b")
axes[1].set_title("learned correction (last fold): sub-additive\nwhen both effects point the same way")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For 1,479 of the 1,520 gene–pair combinations, only one of the two knockdowns acts on the gene
and the additive prediction (of the true singles) is exact. The 41 combinations affected by both carry all of the
error, and most lie closer to zero than the diagonal: combined effects are smaller than the sum — saturation and
masking. With *estimated* single effects the additive baseline's MSE is 0.0171 (0.0124 with the true singles; 0.38 for
predicting no change at all).

On this simulation the **learned model clearly beats the additive baseline**: its error is less than half (MSE 0.0075
vs 0.0171) and it is better on 92 % of the held-out pairs. The contour plot shows what it learned: where both single
effects point in the same direction the correction is sub-additive (red where both are negative, blue where both are
positive); for opposite signs the pattern is more complex, and in the far corners, where there are no training genes
(black dots), it simply extrapolates. It even beats the *measured*
double (0.0091; those cells include partly escaped ones and the mean is noisy), because it applies a smooth rule to
well-estimated single effects.

Be careful what you conclude. Our simulator has **one global interaction rule** — the same saturating function at every
gene, the same masking logic — so a rule learned on 32 pairs transfers well to the other 8. Real genetic
interactions are far more varied and gene-specific (Norman *et al.* 2019 classified synergy, buffering, suppression and
neomorphic behaviour), measured effects are noisier, and most gene pairs do not interact at all. On real Perturb-seq
data, Ahlmann-Eltze, Huber & Anders (2025) found that published deep-learning models did not outperform the additive
baseline for double perturbations, nor simple linear baselines for unseen single perturbations. A learned model is
worth trusting only after it beats strong simple baselines **on held-out perturbations of real data**.

### An unseen single knockdown, from observational data

Suppose gene $T$ was never perturbed. Can we predict $do(T = -2)$ from the control cells alone? If the network were
linear, the total causal effect of $T$ on $Y$ would be the coefficient of $T$ in a regression of $Y$ on $T$ **and the
parents of $T$** — the parents form a valid back-door adjustment set (Section 2). Regressing $Y$ on $T$ alone instead
mixes causation with confounding by $T$'s upstream regulators: genes *upstream* of $T$ would wrongly be predicted to
respond. This is the idea of the **IDA** method (Maathuis *et al.* 2010), which, when the graph is unknown, repeats the
adjustment for every parent set compatible with the observational equivalence class. Here we use the true parents (an
optimistic "oracle" graph) and grade both predictions against the true knockdown effects.

In [ ]:
# ── 7.5 🧪 Predicting each knockdown from control cells: naive regression vs parent adjustment ──
Xc0 = Xc - Xc.mean(axis=0)
rows = []
for t in targets:
    j = gidx[t]
    others = np.setdiff1d(np.arange(n_g), [j])
    parents = np.flatnonzero(A_true[:, j])
    tru, upstream = true_effect.loc[t].to_numpy()[others], ~desc_true[j, others].astype(bool)
    b_naive = Xc0[:, others].T @ Xc0[:, j] / (Xc0[:, j] @ Xc0[:, j])
    b_adj = np.linalg.lstsq(np.column_stack([Xc0[:, j], Xc0[:, parents]]), Xc0[:, others], rcond=None)[0][0]
    for name, b in [("naive regression on T", b_naive), ("adjusted for T's parents", b_adj)]:
        pred = -2.0 * b                                             # linear extrapolation to do(T = −2)
        rows.append({"target": t, "method": name, "correlation with truth": np.corrcoef(pred, tru)[0, 1],
                     "MAE": np.mean(np.abs(pred - tru)), "mean |prediction| for non-descendants": np.mean(np.abs(pred[upstream]))})
ida = pd.DataFrame(rows)
print("🧪 averages over the 15 knocked-down genes (predicted from control cells only):")
print(ida.groupby("method")[["correlation with truth", "MAE", "mean |prediction| for non-descendants"]].mean().round(3).to_string())

🔍 **What just happened?** Adjusting for the parents improves every measure: the predicted effects correlate better
with the true knockdown effects (≈ 0.84 vs 0.74), the error falls, and the spurious "effects" on genes that are *not*
downstream shrink by about a third. They do not vanish: counting noise in the control cells blurs the adjustment, and a
knockdown to −2 is far outside the small fluctuations of control cells, so linear extrapolation misses the saturation.
Observational data can thus say something about interventions — but only through causal assumptions (here, knowing
the parents), and with less accuracy than doing the experiment.

✏️ **Your turn.** Repeat 7.5 on the latent expression `latent[is_ctrl]` (no counting noise). How close do the adjusted
predictions get? What does that tell you about measurement noise as a source of bias in causal estimates?

### 📄 Research connection — Learning and predicting perturbation responses

- **What the field does.** Causal structure learning with interventions goes back to scoring DAGs on mixed data (GIES; Hauser & Bühlmann
    2012) and invariance-based methods; Meinshausen *et al.* (2016) benchmarked causal predictions against held-out yeast
    knockouts, and IDA (Maathuis *et al.* 2010) predicted knockout effects from observational expression data.
    Combinatorial Perturb-seq (Norman *et al.* 2019) mapped genetic interactions. Deep models aim to predict responses
    to unseen perturbations: scGen and the compositional perturbation autoencoder (CPA; Lotfollahi *et al.* 2023) for
    drugs, doses and combinations, and GEARS (Roohani, Huang & Leskovec 2024), which uses a gene-relationship graph to
    generalise to unseen genes and pairs.
- **What this notebook builds.** Descendant sets, transitive reduction and their combination with an observational skeleton, graded against
    the true DAG; an additive baseline and a learned combination rule for double knockdowns, cross-validated over pairs;
    parent adjustment (the IDA idea) for an unseen knockdown.
- **Where the frontier is.** Benchmarks are the frontier as much as models: Ahlmann-Eltze, Huber & Anders (2025) showed that simple
    additive and linear baselines matched or beat several deep-learning and foundation models on public Perturb-seq data,
    which has prompted a re-examination of evaluation practice. Open problems include cycles and feedback, latent
    confounders, cell-state-dependent effects, transfer across cell types, and choosing the next most informative
    experiment (active learning, [Module 29](29_gaussian_processes_bayesian_optimization.ipynb)).
- **Read next.** Norman *et al.* 2019 (*Science*), *Exploring genetic interaction manifolds constructed from rich single-cell phenotypes*; Roohani, Huang & Leskovec 2024 (*Nature Biotechnology*), *GEARS*; Ahlmann-Eltze, Huber & Anders 2025 (*Nature Methods*), linear baselines for perturbation-effect prediction.

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Our screens had 1,000 genes or a 40-gene DAG with an answer key. Here is what changes with real
> perturbation data — no downloads, a map for your first real analysis.

**Scale.** Genome-wide pooled screens target ~20,000 genes with ~4–10 guides each; genome-scale Perturb-seq (Replogle
*et al.* 2022) profiled millions of cells in K562 and RPE1 cells, with a few hundred cells per perturbation; count
matrices are sparse, effect estimation is done on pseudobulk per perturbation or with count GLMs ([Module 24](24_glm_count_data.ipynb)).

**Public data to practise on.** The Replogle *et al.* (2022) genome-scale Perturb-seq data
([gwps.wi.mit.edu](https://gwps.wi.mit.edu/)); the Norman *et al.* (2019) combinatorial CRISPRa screen; DepMap genome-wide
knockout screens across hundreds of cancer cell lines ([depmap.org](https://depmap.org/portal/)); curated collections of
perturbation data sets such as scPerturb ([scperturb.org](https://scperturb.org/)).

**Production tools.** [MAGeCK](https://sourceforge.net/projects/mageck/) and BAGEL for pooled screens; Mixscape in
[Seurat](https://satijalab.org/seurat/) and its Python port in [pertpy](https://pertpy.readthedocs.io/); scGen and CPA
(in pertpy / scvi-tools ecosystems), GEARS; causal-discovery libraries such as causal-learn (Python) and pcalg (R) for PC,
GES/GIES and IDA; DoWhy and EconML for effect estimation in observational data.

**What goes wrong first.**

1. **Off-target effects and guide variability.** Guides can affect other genes (off-target cutting or repression of
   neighbouring genes by CRISPRi), and guides for one gene differ in efficiency; always compare several guides per gene
   and check that the target itself is knocked down.
2. **Cell state and heterogeneity.** Perturbations change the cell cycle, differentiation state or stress responses,
   and cells were heterogeneous to begin with; an average effect can mix very different responses, and
   cell-state-dependent effects are common.
3. **Feedback, compensation and hidden confounders.** Real networks have cycles and adaptive feedback (a knockout can
   be compensated by paralogues within days); unmeasured regulators create dependencies that look like edges; the
   DAG assumptions of Section 6 rarely hold exactly. Treat inferred edges and predicted effects as hypotheses for the
   next experiment.

**What simulation cannot teach.** There is no answer key for real networks; validation uses held-out perturbations
(predict a knockdown before doing it), orthogonal assays (protein levels, imaging), and agreement across cell types
and guides.

In [ ]:
# ── 8.1 Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s on {torch.get_num_threads()} CPU thread(s).")

## 📌 Key takeaways

- **A causal effect is a contrast of potential outcomes**, $\mathbb E[Y(1) - Y(0)]$; associations equal it only when treatment
  is independent of the potential outcomes — which randomisation guarantees and observational data rarely do
  (Simpson's paradox).
- **Causal graphs say what to adjust for:** block back-door paths, never condition on mediators (for total effects) or
  colliders. Regression adjustment needs the outcome model, IPW the propensity model, AIPW either one — and all need
  overlap (positivity) and no unmeasured confounding.
- **Instruments** handle unmeasured confounding if their assumptions hold; weak instruments pull 2SLS towards the
  confounded estimate. Guide assignment in a CRISPR screen is an instrument; escape is non-compliance.
- **Pooled screens** need robust aggregation and honest tests: rogue guides and count noise mislead mean-based
  z-tests; RRA with permutation p-values kept its FDR promise.
- **Perturb-seq** effect estimation needs a composition-robust normalisation and attention to escaping cells; a
  constrained mixture (Mixscape idea) finds them.
- **Networks:** observational data give adjacencies (a moral graph, no directions); interventions give descendants
  and directions; combining them recovers direct edges far better than either.
- **Predicting unseen perturbations** must be judged against strong simple baselines on held-out perturbations: a
  learned combination rule beat additivity in our one-rule simulator, but on real data simple baselines remain hard to
  beat (as of 2025).

---
## ✅ Checkpoint

1. Why does randomisation remove confounding?
   <details><summary>Answer</summary>Assignment by coin makes treatment independent of every patient characteristic, measured or not, and hence of the potential outcomes: E[Y | T = 1] = E[Y(1) | T = 1] = E[Y(1)], and likewise for T = 0, so the difference in means equals the ATE.</details>
2. In the DAG Age → Drug, Age → Outcome, Drug → Biomarker → Outcome, Drug → Outcome, Outcome → Readmission: which variables form a valid back-door adjustment set for the total effect of Drug on Outcome?
   <details><summary>Answer</summary>{Age}. It blocks the only back-door path Drug ← Age → Outcome. Biomarker is a mediator (a descendant of Drug) — adjusting for it removes part of the effect; Readmission is a descendant of the outcome — adjusting for it can induce bias.</details>
3. What does an intervention do to a Bayesian network, and why does that break Markov equivalence?
   <details><summary>Answer</summary>do(X = x) deletes all arrows into X and sets X to x (graph surgery); the rest of the graph is unchanged. Knocking down X changes only X's descendants, so if knocking down A changes B but knocking down B does not change A, the edge is A → B — a distinction the observational distribution cannot make for A → B vs A ← B.</details>
4. By hand: an instrument raises exposure by 0.2 units per allele and the outcome by 0.06 units per allele. What is the Wald estimate, and what would make it biased?
   <details><summary>Answer</summary>0.06 / 0.2 = 0.3 outcome units per exposure unit. It is biased if the variant affects the outcome other than through the exposure (pleiotropy), if it is correlated with confounders (e.g. population structure), or — in finite samples — if the instrument is weak.</details>
5. Why can the mean LFC of a gene's guides call a non-essential gene essential, and how does RRA help?
   <details><summary>Answer</summary>One rogue guide (depleting cells whatever its target) or one noisy low-count guide can pull the mean down. RRA asks whether several guides rank unusually high and uses order statistics with permutation p-values, so one outlier guide does not make a gene significant.</details>
6. A colleague averages all cells with a guide against gene A and reports a 30 % smaller effect than a collaborator who used Mixscape. Who is right?
   <details><summary>Answer</summary>Both estimate different quantities: the average over all guide cells is the intention-to-treat effect, diluted by cells that escaped the knockdown; Mixscape estimates the effect in cells that were actually perturbed (like an effect among compliers). Dividing the ITT effect by the compliance (the target's ITT knockdown relative to full knockdown) reconciles them.</details>
7. Why is the additive baseline for double perturbations hard to beat?
   <details><summary>Answer</summary>Most genes are affected by at most one of the two perturbations, where additivity is nearly exact; genuine interactions are rare, diverse and gene-specific; and single effects are estimated with more cells and less noise than the doubles. A model must learn interaction rules that transfer to unseen pairs — possible in a simulator with one rule, much harder in real biology.</details>
8. In code cell 7.5, what changes between `b_naive` and `b_adj`, and why does it matter for genes upstream of T?
   <details><summary>Answer</summary>b_adj regresses each gene on T *and T's parents* and takes T's coefficient; b_naive ignores the parents. Upstream genes correlate with T through T's parents (a back-door path); adjusting for the parents blocks it, so their predicted response to do(T) moves towards 0, as it should.</details>

## 🧠 Questions & discussion

1. Randomised trials are the gold standard, yet many medical questions are answered with observational data. Which questions can never be randomised, and what would make you trust an observational answer?
2. Our network-learning section assumed no feedback loops. Pick a real regulatory motif with negative feedback (e.g. a transcription factor repressing its own activator). What would a knockdown of each component show, and how would the DAG methods misinterpret it?
3. A model predicts the transcriptomic effect of every unperturbed gene in the genome. What evidence would you require before using its predictions to choose drug targets? How would you design the validation experiment?
4. Genome-scale perturbation atlases are built in a few cell lines (often cancer lines). What can and cannot be transferred to primary human cells or to patients, and who might be harmed if the limits are ignored?
5. Mendelian randomisation, CRISPR screens and randomised trials are all 'natural or designed experiments'. Compare their assumptions: where can each go wrong?

## 🚀 Bonus — going further

- **Interventional structure learning score (GIES idea).** Write a BIC score for a linear-Gaussian DAG that, for each gene, uses only the cells in which that gene was *not* intervened on; compare the scores of the true DAG and of versions with one edge reversed, with and without the interventional cells.
- **Cycles and feedback.** Modify the propagation of `simulate_perturb_seq` (copy it under a new name) to iterate a network with one feedback loop to equilibrium, and re-run Section 6. Which inferred edges become wrong?
- **Optimal experimental design.** Given the observational skeleton, which single gene would you knock down next to orient the most undirected edges? Implement a greedy choice and compare with random choices ([Module 29](29_gaussian_processes_bayesian_optimization.ipynb) treats active learning).
- **A count model for the screen.** Replace the guide LFC test by a negative-binomial GLM of the T0/T1 counts per guide ([Module 24](24_glm_count_data.ipynb)) and aggregate with RRA. Does precision at low coverage improve?
- **Try the real thing.** Install pertpy and run its Mixscape and perturbation-distance tutorials on a public Perturb-seq data set; compare the fraction of escaping cells per guide with what you saw here.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 41 — Perturbation biology and causal inference       Date:
Biological question: when do associations measure causal effects, and what do CRISPR perturbations reveal about gene networks?
Data (SIMULATED): cohort simulators (both potential outcomes known); confounded study (true ATE = 1); eQTL instrument
                  (true θ = 0.5); pooled knockout screen (true essential genes); simulate_perturb_seq (40-gene DAG W,
                  true_effect, latent escape status, 15 singles + 40 doubles)
Method: stratification / regression / IPW / AIPW; Wald and 2SLS; guide LFC + RRA; median-of-ratios + Mixscape-lite;
        partial-correlation skeleton + descendant tests + transitive reduction; additive vs MLP for doubles; IDA idea
Key equation: ATE = E[Y(1) − Y(0)];  P(Y | do(t)) = Σ_s P(Y | t, s) P(s);  θ_Wald = Cov(Z, Y) / Cov(Z, X)
Key parameters: propensity strength β; instrument strength γ (first-stage F); coverage/bottleneck; FDR thresholds
Ground-truth check result: naive vs adjusted ATE …; 2SLS within ±0.25 …; RRA precision/recall …; escape accuracy …;
                           direct-edge precision/recall …; doubles MSE additive vs MLP …
What changed when I modified …: (e.g. β = 4, weak instruments, escape_rate, descendant threshold)
One failure mode: (e.g. positivity violation; collider selection; rogue guides; learned model evaluated on seen pairs)
Connection to research (paper/tool): MAGeCK; Mixscape; GIES/IDA; GEARS/CPA; Ahlmann-Eltze et al. 2025
```

## 📚 References

- Pearl J. (2009) *Causality: Models, Reasoning, and Inference*, 2nd ed. Cambridge University Press.
- Hernán M. A., Robins J. M. (2020) *Causal Inference: What If*. Chapman & Hall/CRC.
- Peters J., Janzing D., Schölkopf B. (2017) *Elements of Causal Inference*. MIT Press.
- Rosenbaum P. R., Rubin D. B. (1983) The central role of the propensity score in observational studies for causal effects. *Biometrika* 70:41–55.
- Angrist J. D., Imbens G. W., Rubin D. B. (1996) Identification of causal effects using instrumental variables. *Journal of the American Statistical Association* 91:444–455.
- Simpson E. H. (1951) The interpretation of interaction in contingency tables. *Journal of the Royal Statistical Society B* 13:238–241.
- Staiger D., Stock J. H. (1997) Instrumental variables regression with weak instruments. *Econometrica* 65:557–586.
- Shalem O. *et al.* (2014) Genome-scale CRISPR-Cas9 knockout screening in human cells. *Science* 343:84–87.
- Wang T., Wei J. J., Sabatini D. M., Lander E. S. (2014) Genetic screens in human cells using the CRISPR-Cas9 system. *Science* 343:80–84.
- Li W. *et al.* (2014) MAGeCK enables robust identification of essential genes from genome-scale CRISPR/Cas9 knockout screens. *Genome Biology* 15:554.
- Kolde R., Laur S., Adler P., Vilo J. (2012) Robust rank aggregation for gene list integration and meta-analysis. *Bioinformatics* 28:573–580.
- Aguirre A. J. *et al.* (2016) Genomic copy number dictates a gene-independent cell response to CRISPR/Cas9 targeting. *Cancer Discovery* 6:914–929.
- Dixit A. *et al.* (2016) Perturb-Seq: dissecting molecular circuits with scalable single-cell RNA profiling of pooled genetic screens. *Cell* 167:1853–1866.
- Adamson B. *et al.* (2016) A multiplexed single-cell CRISPR screening platform enables systematic dissection of the unfolded protein response. *Cell* 167:1867–1882.
- Papalexi E. *et al.* (2021) Characterizing the molecular regulation of inhibitory immune checkpoints with multimodal single-cell screens. *Nature Genetics* 53:322–331.
- Replogle J. M. *et al.* (2022) Mapping information-rich genotype-phenotype landscapes with genome-scale Perturb-seq. *Cell* 185:2559–2575.
- Norman T. M. *et al.* (2019) Exploring genetic interaction manifolds constructed from rich single-cell phenotypes. *Science* 365:786–793.
- Hauser A., Bühlmann P. (2012) Characterization and greedy learning of interventional Markov equivalence classes of directed acyclic graphs. *JMLR* 13:2409–2464.
- Meinshausen N. *et al.* (2016) Methods for causal inference from gene perturbation experiments and validation. *PNAS* 113:7361–7368.
- Maathuis M. H., Colombo D., Kalisch M., Bühlmann P. (2010) Predicting causal effects in large-scale systems from observational data. *Nature Methods* 7:247–248.
- Lotfollahi M. *et al.* (2023) Predicting cellular responses to complex perturbations in high-throughput screens. *Molecular Systems Biology* 19:e11517.
- Roohani Y., Huang K., Leskovec J. (2024) Predicting transcriptional outcomes of novel multigene perturbations with GEARS. *Nature Biotechnology* 42:927–935.
- Ahlmann-Eltze C., Huber W., Anders S. (2025) Deep-learning-based gene perturbation effect prediction does not yet outperform simple linear baselines. *Nature Methods*.

## ➡️ Next

We have learned to ask whether an effect is causal and whether a prediction generalises to perturbations nobody has
done. **[Module 42 · Trustworthy biological ML](42_trustworthy_ml.ipynb)** turns these questions into a toolkit for
every model: detecting leakage, measuring distribution shift, calibrating probabilities and giving predictions with
guaranteed coverage.